In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

titanic_path = kagglehub.competition_download('titanic')

print('Data source import complete.')


## **0. Introduction**

I decided to write this kernel because **Titanic: Machine Learning from Disaster** is one of my favorite competitions on Kaggle. This is a beginner level kernel which focuses on **Exploratory Data Analysis** and **Feature Engineering**. A lot of people start Kaggle with this competition and they get lost in extremely long tutorial kernels. This is a short kernel compared to the other ones. I hope this will be a good guide for starters and inspire them with new feature engineering ideas.

**Titanic: Machine Learning from Disaster** is a great competition to apply domain knowledge for feature engineering, so I made a research and learned a lot about Titanic. There are many secrets to be revealed beneath the Titanic dataset. I tried to find out some of those secret factors that had affected the survival of passengers when the Titanic was sinking. I believe there are other features still waiting to be discovered.

This kernel has **3** main sections; **Exploratory Data Analysis**, **Feature Engineering** and **Model**, and it can achieve top **2%** (**0.83732**) public leaderboard score with a tuned Random Forest Classifier. It takes 60 seconds to run whole notebook. If you have any idea that might improve this kernel, please be sure to comment, or fork and experiment as you like. If you didn't understand any part, feel free to ask.

In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="darkgrid")

from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder, LabelEncoder, StandardScaler
from sklearn.metrics import roc_curve, auc
from sklearn.model_selection import StratifiedKFold

import string
import warnings
warnings.filterwarnings('ignore')

SEED = 42

* Training set has **891** rows and test set has **418** rows
* Training set have **12** features and test set have **11** features
* One extra feature in training set is `Survived` feature, which is the target variable

In [ ]:
import os

def concat_df(train_data, test_data):
    # Returns a concatenated df of training and test set
    return pd.concat([train_data, test_data], sort=True).reset_index(drop=True)

def divide_df(all_data):
    # Returns divided dfs of training and test set
    return all_data.loc[:890], all_data.loc[891:].drop(['Survived'], axis=1)

# Construct the correct paths using titanic_path
train_csv_path = os.path.join(titanic_path, 'train.csv')
test_csv_path = os.path.join(titanic_path, 'test.csv')

df_train = pd.read_csv(train_csv_path)
df_test = pd.read_csv(test_csv_path)
df_all = concat_df(df_train, df_test)

df_train.name = 'Training Set'
df_test.name = 'Test Set'
df_all.name = 'All Set'

dfs = [df_train, df_test]

print('Number of Training Examples = {}'.format(df_train.shape[0]))
print('Number of Test Examples = {}\n'.format(df_test.shape[0]))
print('Training X Shape = {}'.format(df_train.shape))
print('Training y Shape = {}\n'.format(df_train['Survived'].shape[0]))
print('Test X Shape = {}'.format(df_test.shape))
print('Test y Shape = {}\n'.format(df_test.shape[0]))
print(df_train.columns)
print(df_test.columns)

## **1. Exploratory Data Analysis**

### **1.1 Overview**
* `PassengerId` is the unique id of the row and it doesn't have any effect on target
* `Survived` is the target variable we are trying to predict (**0** or **1**):
    - **1 = Survived**
    - **0 = Not Survived**
* `Pclass` (Passenger Class) is the socio-economic status of the passenger and it is a categorical ordinal feature which has **3** unique values (**1**,  **2 **or **3**):
    - **1 = Upper Class**
    - **2 = Middle Class**
    - **3 = Lower Class**
* `Name`, `Sex` and `Age` are self-explanatory
* `SibSp` is the total number of the passengers' siblings and spouse
* `Parch` is the total number of the passengers' parents and children
* `Ticket` is the ticket number of the passenger
* `Fare` is the passenger fare
* `Cabin` is the cabin number of the passenger
* `Embarked` is port of embarkation and it is a categorical feature which has **3** unique values (**C**, **Q** or **S**):
    - **C = Cherbourg**
    - **Q = Queenstown**
    - **S = Southampton**

In [ ]:
print(df_train.info())
df_train.sample(3)

In [ ]:
print(df_test.info())
df_test.sample(3)

### **1.2 Missing Values**
As seen from below, some columns have missing values. `display_missing` function shows the count of missing values in every column in both training and test set.
* Training set have missing values in `Age`, `Cabin` and `Embarked` columns
* Test set have missing values in `Age`, `Cabin` and `Fare` columns

It is convenient to work on concatenated training and test set while dealing with missing values, otherwise filled data may overfit to training or test set samples. The count of missing values in `Age`, `Embarked` and `Fare` are smaller compared to total sample, but roughly **80%** of the `Cabin` is missing. Missing values in `Age`, `Embarked` and `Fare` can be filled with descriptive statistical measures but that wouldn't work for `Cabin`.

In [ ]:
def display_missing(df):
    for col in df.columns.tolist():
        print('{} column missing values: {}'.format(col, df[col].isnull().sum()))
    print('\n')

for df in dfs:
    print('{}'.format(df.name))
    display_missing(df)

#### **1.2.1 Age**
Missing values in `Age` are filled with median age, but using median age of the whole data set is not a good choice. Median age of `Pclass` groups is the best choice because of its high correlation with `Age` **(0.408106)** and `Survived` **(0.338481)**. It is also more logical to group ages by passenger classes instead of other features.

In [ ]:
df_all_corr = df_all.drop('Cabin', axis=1).corr(numeric_only=True).abs().unstack().sort_values(kind="quicksort", ascending=False).reset_index()
df_all_corr.rename(columns={"level_0": "Feature 1", "level_1": "Feature 2", 0: 'Correlation Coefficient'}, inplace=True)
df_all_corr[df_all_corr['Feature 1'] == 'Age']

In order to be more accurate, `Sex` feature is used as the second level of `groupby` while filling the missing `Age` values. As seen from below, `Pclass` and `Sex` groups have distinct median `Age` values. When passenger class increases, the median age for both males and females also increases. However, females tend to have slightly lower median `Age` than males. The median ages below are used for filling the missing values in `Age` feature.

In [ ]:
age_by_pclass_sex = df_all.groupby(['Sex', 'Pclass']).median(numeric_only=True)['Age']

for pclass in range(1, 4):
    for sex in ['female', 'male']:
        print('Median age of Pclass {} {}s: {}'.format(pclass, sex, age_by_pclass_sex[sex][pclass]))
print('Median age of all passengers: {}'.format(df_all['Age'].median(numeric_only=True)))

# Filling the missing values in Age with the medians of Sex and Pclass groups
df_all['Age'] = df_all.groupby(['Sex', 'Pclass'])['Age'].transform(lambda x: x.fillna(x.median(numeric_only=True)))

#### **1.2.2 Embarked**
`Embarked` is a categorical feature and there are only **2** missing values in whole data set. Both of those passengers are female, upper class and they have the same ticket number. This means that they know each other and embarked from the same port together. The mode `Embarked` value for an upper class female passenger is **C (Cherbourg)**, but this doesn't necessarily mean that they embarked from that port.

In [ ]:
df_all[df_all['Embarked'].isnull()]

When I googled **Stone, Mrs. George Nelson (Martha Evelyn)**, I found that she embarked from **S (Southampton)** with her maid **Amelie Icard**, in this page [Martha Evelyn Stone: Titanic Survivor](https://www.encyclopedia-titanica.org/titanic-survivor/martha-evelyn-stone.html).

> *Mrs Stone boarded the Titanic in Southampton on 10 April 1912 and was travelling in first class with her maid Amelie Icard. She occupied cabin B-28.*

Missing values in `Embarked` are filled with **S** with this information.

In [ ]:
# Filling the missing values in Embarked with S
df_all['Embarked'] = df_all['Embarked'].fillna('S')

#### **1.2.3 Fare**
There is only one passenger with missing `Fare` value. We can assume that `Fare` is related to family size (`Parch` and `SibSp`) and `Pclass` features. Median `Fare` value of a male with a third class ticket and no family is a logical choice to fill the missing value.

In [ ]:
df_all[df_all['Fare'].isnull()]

In [ ]:
med_fare = df_all.groupby(['Pclass', 'Parch', 'SibSp']).Fare.median()[3][0][0]
# Filling the missing value in Fare with the median Fare of 3rd class alone passenger
df_all['Fare'] = df_all['Fare'].fillna(med_fare)

#### **1.2.4 Cabin**
`Cabin` feature is little bit tricky and it needs further exploration. The large portion of the `Cabin` feature is missing and the feature itself can't be ignored completely because some the cabins might have higher survival rates. It turns out to be the first letter of the `Cabin` values are the decks in which the cabins are located. Those decks were mainly separated for one passenger class, but some of them were used by multiple passenger classes.
![alt text](https://vignette.wikia.nocookie.net/titanic/images/f/f9/Titanic_side_plan.png/revision/latest?cb=20180322183733)
* On the Boat Deck there were **6** rooms labeled as **T, U, W, X, Y, Z** but only the **T** cabin is present in the dataset
* **A**, **B** and **C** decks were only for 1st class passengers
* **D** and **E** decks were for all classes
* **F** and **G** decks were for both 2nd and 3rd class passengers
* From going **A** to **G**, distance to the staircase increases which might be a factor of survival

In [ ]:
# Creating Deck column from the first letter of the Cabin column (M stands for Missing)
df_all['Deck'] = df_all['Cabin'].apply(lambda s: s[0] if pd.notnull(s) else 'M')

df_all_decks = df_all.groupby(['Deck', 'Pclass']).count().drop(columns=['Survived', 'Sex', 'Age', 'SibSp', 'Parch',
                                                                        'Fare', 'Embarked', 'Cabin', 'PassengerId', 'Ticket']).rename(columns={'Name': 'Count'}).transpose()

def get_pclass_dist(df):

    # Creating a dictionary for every passenger class count in every deck
    deck_counts = {'A': {}, 'B': {}, 'C': {}, 'D': {}, 'E': {}, 'F': {}, 'G': {}, 'M': {}, 'T': {}}
    decks = df.columns.levels[0]

    for deck in decks:
        for pclass in range(1, 4):
            try:
                count = df[deck][pclass][0]
                deck_counts[deck][pclass] = count
            except KeyError:
                deck_counts[deck][pclass] = 0

    df_decks = pd.DataFrame(deck_counts)
    deck_percentages = {}

    # Creating a dictionary for every passenger class percentage in every deck
    for col in df_decks.columns:
        deck_percentages[col] = [(count / df_decks[col].sum()) * 100 for count in df_decks[col]]

    return deck_counts, deck_percentages

def display_pclass_dist(percentages):

    df_percentages = pd.DataFrame(percentages).transpose()
    deck_names = ('A', 'B', 'C', 'D', 'E', 'F', 'G', 'M', 'T')
    bar_count = np.arange(len(deck_names))
    bar_width = 0.85

    pclass1 = df_percentages[0]
    pclass2 = df_percentages[1]
    pclass3 = df_percentages[2]

    plt.figure(figsize=(20, 10))
    plt.bar(bar_count, pclass1, color='#b5ffb9', edgecolor='white', width=bar_width, label='Passenger Class 1')
    plt.bar(bar_count, pclass2, bottom=pclass1, color='#f9bc86', edgecolor='white', width=bar_width, label='Passenger Class 2')
    plt.bar(bar_count, pclass3, bottom=pclass1 + pclass2, color='#a3acff', edgecolor='white', width=bar_width, label='Passenger Class 3')

    plt.xlabel('Deck', size=15, labelpad=20)
    plt.ylabel('Passenger Class Percentage', size=15, labelpad=20)
    plt.xticks(bar_count, deck_names)
    plt.tick_params(axis='x', labelsize=15)
    plt.tick_params(axis='y', labelsize=15)

    plt.legend(loc='upper left', bbox_to_anchor=(1, 1), prop={'size': 15})
    plt.title('Passenger Class Distribution in Decks', size=18, y=1.05)

    plt.show()

all_deck_count, all_deck_per = get_pclass_dist(df_all_decks)
display_pclass_dist(all_deck_per)

* **100%** of **A**, **B** and **C** decks are 1st class passengers
* Deck **D** has **87%** 1st class and **13%** 2nd class passengers
* Deck **E** has **83%** 1st class, **10%** 2nd class and **7%** 3rd class passengers
* Deck **F** has **62%** 2nd class and **38%** 3rd class passengers
* **100%** of **G** deck are 3rd class passengers
* There is one person on the boat deck in **T** cabin and he is a 1st class passenger. **T** cabin passenger has the closest resemblance to **A** deck passengers so he is grouped with **A** deck
* Passengers labeled as **M** are the missing values in `Cabin` feature. I don't think it is possible to find those passengers' real `Deck` so I decided to use **M** like a deck

In [ ]:
# Passenger in the T deck is changed to A
idx = df_all[df_all['Deck'] == 'T'].index
df_all.loc[idx, 'Deck'] = 'A'

In [ ]:
df_all_decks_survived = df_all.groupby(['Deck', 'Survived']).count().drop(columns=['Sex', 'Age', 'SibSp', 'Parch', 'Fare',
                                                                                   'Embarked', 'Pclass', 'Cabin', 'PassengerId', 'Ticket']).rename(columns={'Name':'Count'}).transpose()

def get_survived_dist(df):

    # Creating a dictionary for every survival count in every deck
    surv_counts = {'A':{}, 'B':{}, 'C':{}, 'D':{}, 'E':{}, 'F':{}, 'G':{}, 'M':{}}
    decks = df.columns.levels[0]

    for deck in decks:
        for survive in range(0, 2):
            surv_counts[deck][survive] = df[deck][survive][0]

    df_surv = pd.DataFrame(surv_counts)
    surv_percentages = {}

    for col in df_surv.columns:
        surv_percentages[col] = [(count / df_surv[col].sum()) * 100 for count in df_surv[col]]

    return surv_counts, surv_percentages

def display_surv_dist(percentages):

    df_survived_percentages = pd.DataFrame(percentages).transpose()
    deck_names = ('A', 'B', 'C', 'D', 'E', 'F', 'G', 'M')
    bar_count = np.arange(len(deck_names))
    bar_width = 0.85

    not_survived = df_survived_percentages[0]
    survived = df_survived_percentages[1]

    plt.figure(figsize=(20, 10))
    plt.bar(bar_count, not_survived, color='#b5ffb9', edgecolor='white', width=bar_width, label="Not Survived")
    plt.bar(bar_count, survived, bottom=not_survived, color='#f9bc86', edgecolor='white', width=bar_width, label="Survived")

    plt.xlabel('Deck', size=15, labelpad=20)
    plt.ylabel('Survival Percentage', size=15, labelpad=20)
    plt.xticks(bar_count, deck_names)
    plt.tick_params(axis='x', labelsize=15)
    plt.tick_params(axis='y', labelsize=15)

    plt.legend(loc='upper left', bbox_to_anchor=(1, 1), prop={'size': 15})
    plt.title('Survival Percentage in Decks', size=18, y=1.05)

    plt.show()

all_surv_count, all_surv_per = get_survived_dist(df_all_decks_survived)
display_surv_dist(all_surv_per)

As I suspected, every deck has different survival rates and that information can't be discarded. Deck **B**, **C**, **D** and **E** have the highest survival rates. Those decks are mostly occupied by 1st class passengers. **M** has the lowest survival rate which is mostly occupied by 2nd and 3rd class passengers. To conclude, cabins used by 1st class passengers have higher survival rates than cabins used by 2nd and 3rd class passengers. In my opinion **M** (Missing `Cabin` values) has the lowest survival rate because they couldn't retrieve the cabin data of the victims. That's why I believe labeling that group as **M** is a reasonable way to handle the missing data. It is a unique group with shared characteristics. `Deck` feature has high-cardinality right now so some of the values are grouped with each other based on their similarities.
* **A**, **B** and **C** decks are labeled as **ABC** because all of them have only 1st class passengers
* **D** and **E** decks are labeled as **DE** because both of them have similar passenger class distribution and same survival rate
* **F** and **G** decks are labeled as **FG** because of the same reason above
* **M** deck doesn't need to be grouped with other decks because it is very different from others and has the lowest survival rate.

In [ ]:
df_all['Deck'] = df_all['Deck'].replace(['A', 'B', 'C'], 'ABC')
df_all['Deck'] = df_all['Deck'].replace(['D', 'E'], 'DE')
df_all['Deck'] = df_all['Deck'].replace(['F', 'G'], 'FG')

df_all['Deck'].value_counts()

After filling the missing values in `Age`, `Embarked`, `Fare` and `Deck` features, there is no missing value left in both training and test set. `Cabin` is dropped because `Deck` feature is used instead of it.

In [ ]:
# Dropping the Cabin feature
df_all.drop(['Cabin'], inplace=True, axis=1)

df_train, df_test = divide_df(df_all)
dfs = [df_train, df_test]

for df in dfs:
    display_missing(df)

### **1.3 Target Distribution**
* **38.38%** (342/891) of training set is **Class 1**
* **61.62%** (549/891) of training set is **Class 0**

In [ ]:
survived = df_train['Survived'].value_counts()[1]
not_survived = df_train['Survived'].value_counts()[0]
survived_per = survived / df_train.shape[0] * 100
not_survived_per = not_survived / df_train.shape[0] * 100

print('{} of {} passengers survived and it is the {:.2f}% of the training set.'.format(survived, df_train.shape[0], survived_per))
print('{} of {} passengers didnt survive and it is the {:.2f}% of the training set.'.format(not_survived, df_train.shape[0], not_survived_per))

plt.figure(figsize=(10, 8))
sns.countplot(df_train['Survived'])

plt.xlabel('Survival', size=15, labelpad=15)
plt.ylabel('Passenger Count', size=15, labelpad=15)
plt.xticks((0, 1), ['Not Survived ({0:.2f}%)'.format(not_survived_per), 'Survived ({0:.2f}%)'.format(survived_per)])
plt.tick_params(axis='x', labelsize=13)
plt.tick_params(axis='y', labelsize=13)

plt.title('Training Set Survival Distribution', size=15, y=1.05)

plt.show()

### **1.4 Correlations**
Features are highly correlated with each other and dependent to each other. The highest correlation between features is **0.549500** in training set and **0.577147** in test set (between `Fare` and `Pclass`). The other features are also highly correlated. There are **9** correlations in training set and **6** correlations in test set that are higher than **0.1**.

In [ ]:
df_train_corr = df_train.drop(['PassengerId'], axis=1).corr(numeric_only=True).abs().unstack().sort_values(kind="quicksort", ascending=False).reset_index()
df_train_corr.rename(columns={"level_0": "Feature 1", "level_1": "Feature 2", 0: 'Correlation Coefficient'}, inplace=True)
df_train_corr.drop(df_train_corr.iloc[1::2].index, inplace=True)
df_train_corr_nd = df_train_corr.drop(df_train_corr[df_train_corr['Correlation Coefficient'] == 1.0].index)

df_test_corr = df_test.corr(numeric_only=True).abs().unstack().sort_values(kind="quicksort", ascending=False).reset_index()
df_test_corr.rename(columns={"level_0": "Feature 1", "level_1": "Feature 2", 0: 'Correlation Coefficient'}, inplace=True)
df_test_corr.drop(df_test_corr.iloc[1::2].index, inplace=True)
df_test_corr_nd = df_test_corr.drop(df_test_corr[df_test_corr['Correlation Coefficient'] == 1.0].index)

In [ ]:
# Training set high correlations
corr = df_train_corr_nd['Correlation Coefficient'] > 0.1
df_train_corr_nd[corr]

In [ ]:
# Test set high correlations
corr = df_test_corr_nd['Correlation Coefficient'] > 0.1
df_test_corr_nd[corr]

In [ ]:
fig, axs = plt.subplots(nrows=2, figsize=(20, 20))

sns.heatmap(df_train.drop(['PassengerId'], axis=1).corr(numeric_only=True), ax=axs[0], annot=True, square=True, cmap='coolwarm', annot_kws={'size': 14})
sns.heatmap(df_test.drop(['PassengerId'], axis=1).corr(numeric_only=True), ax=axs[1], annot=True, square=True, cmap='coolwarm', annot_kws={'size': 14})

for i in range(2):
    axs[i].tick_params(axis='x', labelsize=14)
    axs[i].tick_params(axis='y', labelsize=14)

axs[0].set_title('Training Set Correlations', size=15)
axs[1].set_title('Test Set Correlations', size=15)

plt.show()

### **1.5 Target Distribution in Features**

#### **1.5.1 Continuous Features**
Both of the continuous features (`Age` and `Fare`) have good split points and spikes for a decision tree to learn. One potential problem for both features is, the distribution has more spikes and bumps in training set, but it is smoother in test set. Model may not be able to generalize to test set because of this reason.

* Distribution of `Age` feature clearly shows that children younger than 15 has a higher survival rate than any of the other age groups
* In distribution of `Fare` feature, the survival rate is higher on distribution tails. The distribution also has positive skew because of the extremely large outliers

In [ ]:
cont_features = ['Age', 'Fare']
surv = df_train['Survived'] == 1

fig, axs = plt.subplots(ncols=2, nrows=2, figsize=(20, 20))
plt.subplots_adjust(right=1.5)

for i, feature in enumerate(cont_features):
    # Distribution of survival in feature
    sns.distplot(df_train[~surv][feature], label='Not Survived', hist=True, color='#e74c3c', ax=axs[0][i])
    sns.distplot(df_train[surv][feature], label='Survived', hist=True, color='#2ecc71', ax=axs[0][i])

    # Distribution of feature in dataset
    sns.distplot(df_train[feature], label='Training Set', hist=False, color='#e74c3c', ax=axs[1][i])
    sns.distplot(df_test[feature], label='Test Set', hist=False, color='#2ecc71', ax=axs[1][i])

    axs[0][i].set_xlabel('')
    axs[1][i].set_xlabel('')

    for j in range(2):
        axs[i][j].tick_params(axis='x', labelsize=20)
        axs[i][j].tick_params(axis='y', labelsize=20)

    axs[0][i].legend(loc='upper right', prop={'size': 20})
    axs[1][i].legend(loc='upper right', prop={'size': 20})
    axs[0][i].set_title('Distribution of Survival in {}'.format(feature), size=20, y=1.05)

axs[1][0].set_title('Distribution of {} Feature'.format('Age'), size=20, y=1.05)
axs[1][1].set_title('Distribution of {} Feature'.format('Fare'), size=20, y=1.05)

plt.show()

#### **1.5.2 Categorical Features**
Every categorical feature has at least one class with high mortality rate. Those classes are very helpful to predict whether the passenger is a survivor or victim. Best categorical features are `Pclass` and `Sex` because they have the most homogenous distributions.

* Passengers boarded from **Southampton** has a lower survival rate unlike other ports. More than half of the passengers boarded from **Cherbourg** had survived. This observation could be related to `Pclass` feature
* `Parch` and `SibSp` features show that passengers with only one family member has a higher survival rate

In [ ]:
cat_features = ['Embarked', 'Parch', 'Pclass', 'Sex', 'SibSp', 'Deck']

fig, axs = plt.subplots(ncols=2, nrows=3, figsize=(20, 20))
plt.subplots_adjust(right=1.5, top=1.25)

for i, feature in enumerate(cat_features, 1):
    plt.subplot(2, 3, i)
    sns.countplot(x=feature, hue='Survived', data=df_train)

    plt.xlabel('{}'.format(feature), size=20, labelpad=15)
    plt.ylabel('Passenger Count', size=20, labelpad=15)
    plt.tick_params(axis='x', labelsize=20)
    plt.tick_params(axis='y', labelsize=20)

    plt.legend(['Not Survived', 'Survived'], loc='upper center', prop={'size': 18})
    plt.title('Count of Survival in {} Feature'.format(feature), size=20, y=1.05)

plt.show()

### **1.6 Conclusion**
Most of the features are correlated with each other. This relationship can be used to create new features with feature transformation and feature interaction. Target encoding could be very useful as well because of the high correlations with `Survived` feature.

Split points and spikes are visible in continuous features. They can be captured easily with a decision tree model, but linear models may not be able to spot them.

Categorical features have very distinct distributions with different survival rates. Those features can be one-hot encoded. Some of those features may be combined with each other to make new features.

Created a new feature called `Deck` and dropped `Cabin` feature at the **Exploratory Data Analysis** part.

In [ ]:
df_all = concat_df(df_train, df_test)
df_all.head()

## **2. Feature Engineering**

### **2.1 Binning Continuous Features**

#### **2.1.1 Fare**
`Fare` feature is positively skewed and survival rate is extremely high on the right end. **13** quantile based bins are used for `Fare` feature. Even though the bins are too much, they provide decent amount of information gain. The groups at the left side of the graph has the lowest survival rate and the groups at the right side of the graph has the highest survival rate. This high survival rate was not visible in the distribution graph. There is also an unusual group **(15.742, 23.25]** in the middle with high survival rate that is captured in this process.

In [ ]:
df_all['Fare'] = pd.qcut(df_all['Fare'], 13)

In [ ]:
fig, axs = plt.subplots(figsize=(22, 9))
sns.countplot(x='Fare', hue='Survived', data=df_all)

plt.xlabel('Fare', size=15, labelpad=20)
plt.ylabel('Passenger Count', size=15, labelpad=20)
plt.tick_params(axis='x', labelsize=10)
plt.tick_params(axis='y', labelsize=15)

plt.legend(['Not Survived', 'Survived'], loc='upper right', prop={'size': 15})
plt.title('Count of Survival in {} Feature'.format('Fare'), size=15, y=1.05)

plt.show()

#### **2.1.2 Age**
`Age` feature has a normal distribution with some spikes and bumps and **10** quantile based bins are used for `Age`. The first bin has the highest survival rate and 4th bin has the lowest survival rate. Those were the biggest spikes in the distribution. There is also an unusual group **(34.0, 40.0]** with high survival rate that is captured in this process.

In [ ]:
df_all['Age'] = pd.qcut(df_all['Age'], 10)

In [ ]:
fig, axs = plt.subplots(figsize=(22, 9))
sns.countplot(x='Age', hue='Survived', data=df_all)

plt.xlabel('Age', size=15, labelpad=20)
plt.ylabel('Passenger Count', size=15, labelpad=20)
plt.tick_params(axis='x', labelsize=15)
plt.tick_params(axis='y', labelsize=15)

plt.legend(['Not Survived', 'Survived'], loc='upper right', prop={'size': 15})
plt.title('Survival Counts in {} Feature'.format('Age'), size=15, y=1.05)

plt.show()

### **2.2 Frequency Encoding**
`Family_Size` is created by adding `SibSp`, `Parch` and **1**. `SibSp` is the count of siblings and spouse, and `Parch` is the count of parents and children. Those columns are added in order to find the total size of families. Adding **1** at the end, is the current passenger. Graphs have clearly shown that family size is a predictor of survival because different values have different survival rates.
* Family Size with **1** are labeled as **Alone**
* Family Size with **2**, **3** and **4** are labeled as **Small**
* Family Size with **5** and **6** are labeled as **Medium**
* Family Size with **7**, **8** and **11** are labeled as **Large**

In [ ]:
df_all['Family_Size'] = df_all['SibSp'] + df_all['Parch'] + 1

fig, axs = plt.subplots(figsize=(20, 20), ncols=2, nrows=2)
plt.subplots_adjust(right=1.5)

sns.barplot(x=df_all['Family_Size'].value_counts().index, y=df_all['Family_Size'].value_counts().values, ax=axs[0][0])
sns.countplot(x='Family_Size', hue='Survived', data=df_all, ax=axs[0][1])

axs[0][0].set_title('Family Size Feature Value Counts', size=20, y=1.05)
axs[0][1].set_title('Survival Counts in Family Size ', size=20, y=1.05)

family_map = {1: 'Alone', 2: 'Small', 3: 'Small', 4: 'Small', 5: 'Medium', 6: 'Medium', 7: 'Large', 8: 'Large', 11: 'Large'}
df_all['Family_Size_Grouped'] = df_all['Family_Size'].map(family_map)

sns.barplot(x=df_all['Family_Size_Grouped'].value_counts().index, y=df_all['Family_Size_Grouped'].value_counts().values, ax=axs[1][0])
sns.countplot(x='Family_Size_Grouped', hue='Survived', data=df_all, ax=axs[1][1])

axs[1][0].set_title('Family Size Feature Value Counts After Grouping', size=20, y=1.05)
axs[1][1].set_title('Survival Counts in Family Size After Grouping', size=20, y=1.05)

for i in range(2):
    axs[i][1].legend(['Not Survived', 'Survived'], loc='upper right', prop={'size': 20})
    for j in range(2):
        axs[i][j].tick_params(axis='x', labelsize=20)
        axs[i][j].tick_params(axis='y', labelsize=20)
        axs[i][j].set_xlabel('')
        axs[i][j].set_ylabel('')

plt.show()

There are too many unique `Ticket` values to analyze, so grouping them up by their frequencies makes things easier.

**How is this feature different than `Family_Size`?** Many passengers travelled along with groups. Those groups consist of friends, nannies, maids and etc. They weren't counted as family, but they used the same ticket.

**Why not grouping tickets by their prefixes?** If prefixes in `Ticket` feature has any meaning, then they are already captured in `Pclass` or `Embarked` features because that could be the only logical information which can be derived from the `Ticket` feature.

According to the graph below, groups with **2**,**3** and **4** members had a higher survival rate. Passengers who travel alone has the lowest survival rate. After **4** group members, survival rate decreases drastically. This pattern is very similar to `Family_Size` feature but there are minor differences. `Ticket_Frequency` values are not grouped like `Family_Size` because that would basically create the same feature with perfect correlation. This kind of feature wouldn't provide any additional information gain.

In [ ]:
df_all['Ticket_Frequency'] = df_all.groupby('Ticket')['Ticket'].transform('count')

In [ ]:
fig, axs = plt.subplots(figsize=(12, 9))
sns.countplot(x='Ticket_Frequency', hue='Survived', data=df_all)

plt.xlabel('Ticket Frequency', size=15, labelpad=20)
plt.ylabel('Passenger Count', size=15, labelpad=20)
plt.tick_params(axis='x', labelsize=15)
plt.tick_params(axis='y', labelsize=15)

plt.legend(['Not Survived', 'Survived'], loc='upper right', prop={'size': 15})
plt.title('Count of Survival in {} Feature'.format('Ticket Frequency'), size=15, y=1.05)

plt.show()

### **2.3 Title & Is Married**
`Title` is created by extracting the prefix before `Name` feature. According to graph below, there are many titles that are occuring very few times. Some of those titles doesn't seem correct and they need to be replaced. **Miss**, **Mrs**, **Ms**, **Mlle**, **Lady**, **Mme**, **the Countess**, **Dona** titles are replaced with **Miss/Mrs/Ms** because all of them are female. Values like **Mlle**, **Mme** and **Dona** are actually the name of the passengers, but they are classified as titles because `Name` feature is split by comma. **Dr**, **Col**, **Major**, **Jonkheer**, **Capt**, **Sir**, **Don** and **Rev** titles are replaced with **Dr/Military/Noble/Clergy** because those passengers have similar characteristics. **Master** is a unique title. It is given to male passengers below age **26**. They have the highest survival rate among all males.

`Is_Married` is a binary feature based on the **Mrs** title. **Mrs** title has the highest survival rate among other female titles. This title needs to be a feature because all female titles are grouped with each other.

In [ ]:
df_all['Title'] = df_all['Name'].str.split(', ', expand=True)[1].str.split('.', expand=True)[0]
df_all['Is_Married'] = 0
df_all['Is_Married'].loc[df_all['Title'] == 'Mrs'] = 1

In [ ]:
fig, axs = plt.subplots(nrows=2, figsize=(20, 20))
sns.barplot(x=df_all['Title'].value_counts().index, y=df_all['Title'].value_counts().values, ax=axs[0])

axs[0].tick_params(axis='x', labelsize=10)
axs[1].tick_params(axis='x', labelsize=15)

for i in range(2):
    axs[i].tick_params(axis='y', labelsize=15)

axs[0].set_title('Title Feature Value Counts', size=20, y=1.05)

df_all['Title'] = df_all['Title'].replace(['Miss', 'Mrs','Ms', 'Mlle', 'Lady', 'Mme', 'the Countess', 'Dona'], 'Miss/Mrs/Ms')
df_all['Title'] = df_all['Title'].replace(['Dr', 'Col', 'Major', 'Jonkheer', 'Capt', 'Sir', 'Don', 'Rev'], 'Dr/Military/Noble/Clergy')

sns.barplot(x=df_all['Title'].value_counts().index, y=df_all['Title'].value_counts().values, ax=axs[1])
axs[1].set_title('Title Feature Value Counts After Grouping', size=20, y=1.05)

plt.show()

### **2.4 Target Encoding**
`extract_surname` function is used for extracting surnames of passengers from the `Name` feature. `Family` feature is created with the extracted surname. This is necessary for grouping passengers in the same family.

In [ ]:
def extract_surname(data):

    families = []

    for i in range(len(data)):
        name = data.iloc[i]

        if '(' in name:
            name_no_bracket = name.split('(')[0]
        else:
            name_no_bracket = name

        family = name_no_bracket.split(',')[0]
        title = name_no_bracket.split(',')[1].strip().split(' ')[0]

        for c in string.punctuation:
            family = family.replace(c, '').strip()

        families.append(family)

    return families

df_all['Family'] = extract_surname(df_all['Name'])
df_train = df_all.loc[:890]
df_test = df_all.loc[891:]
dfs = [df_train, df_test]

`Family_Survival_Rate` is calculated from families in training set since there is no `Survived` feature in test set. A list of family names that are occuring in both training and test set (`non_unique_families`), is created. The survival rate is calculated for families with more than 1 members in that list, and stored in `Family_Survival_Rate` feature.

An extra binary feature `Family_Survival_Rate_NA` is created for families that are unique to the test set. This feature is also necessary because there is no way to calculate those families' survival rate. This feature implies that family survival rate is not applicable to those passengers because there is no way to retrieve their survival rate.

`Ticket_Survival_Rate` and `Ticket_Survival_Rate_NA` features are also created with the same method. `Ticket_Survival_Rate` and `Family_Survival_Rate` are averaged and become `Survival_Rate`, and `Ticket_Survival_Rate_NA` and `Family_Survival_Rate_NA` are also averaged and become `Survival_Rate_NA`.

In [ ]:
# Creating a list of families and tickets that are occuring in both training and test set
non_unique_families = [x for x in df_train['Family'].unique() if x in df_test['Family'].unique()]
non_unique_tickets = [x for x in df_train['Ticket'].unique() if x in df_test['Ticket'].unique()]

df_family_survival_rate = df_train.groupby('Family')[['Survived','Family_Size']].median()
df_ticket_survival_rate = df_train.groupby('Ticket')[['Survived','Ticket_Frequency']].median()

family_rates = {}
ticket_rates = {}

for i in range(len(df_family_survival_rate)):
    # Checking a family exists in both training and test set, and has members more than 1
    if df_family_survival_rate.index[i] in non_unique_families and df_family_survival_rate.iloc[i, 1] > 1:
        family_rates[df_family_survival_rate.index[i]] = df_family_survival_rate.iloc[i, 0]

for i in range(len(df_ticket_survival_rate)):
    # Checking a ticket exists in both training and test set, and has members more than 1
    if df_ticket_survival_rate.index[i] in non_unique_tickets and df_ticket_survival_rate.iloc[i, 1] > 1:
        ticket_rates[df_ticket_survival_rate.index[i]] = df_ticket_survival_rate.iloc[i, 0]

In [ ]:
mean_survival_rate = np.mean(df_train['Survived'])

train_family_survival_rate = []
train_family_survival_rate_NA = []
test_family_survival_rate = []
test_family_survival_rate_NA = []

for i in range(len(df_train)):
    if df_train['Family'][i] in family_rates:
        train_family_survival_rate.append(family_rates[df_train['Family'][i]])
        train_family_survival_rate_NA.append(1)
    else:
        train_family_survival_rate.append(mean_survival_rate)
        train_family_survival_rate_NA.append(0)

for i in range(len(df_test)):
    if df_test['Family'].iloc[i] in family_rates:
        test_family_survival_rate.append(family_rates[df_test['Family'].iloc[i]])
        test_family_survival_rate_NA.append(1)
    else:
        test_family_survival_rate.append(mean_survival_rate)
        test_family_survival_rate_NA.append(0)

df_train['Family_Survival_Rate'] = train_family_survival_rate
df_train['Family_Survival_Rate_NA'] = train_family_survival_rate_NA
df_test['Family_Survival_Rate'] = test_family_survival_rate
df_test['Family_Survival_Rate_NA'] = test_family_survival_rate_NA

train_ticket_survival_rate = []
train_ticket_survival_rate_NA = []
test_ticket_survival_rate = []
test_ticket_survival_rate_NA = []

for i in range(len(df_train)):
    if df_train['Ticket'][i] in ticket_rates:
        train_ticket_survival_rate.append(ticket_rates[df_train['Ticket'][i]])
        train_ticket_survival_rate_NA.append(1)
    else:
        train_ticket_survival_rate.append(mean_survival_rate)
        train_ticket_survival_rate_NA.append(0)

for i in range(len(df_test)):
    if df_test['Ticket'].iloc[i] in ticket_rates:
        test_ticket_survival_rate.append(ticket_rates[df_test['Ticket'].iloc[i]])
        test_ticket_survival_rate_NA.append(1)
    else:
        test_ticket_survival_rate.append(mean_survival_rate)
        test_ticket_survival_rate_NA.append(0)

df_train['Ticket_Survival_Rate'] = train_ticket_survival_rate
df_train['Ticket_Survival_Rate_NA'] = train_ticket_survival_rate_NA
df_test['Ticket_Survival_Rate'] = test_ticket_survival_rate
df_test['Ticket_Survival_Rate_NA'] = test_ticket_survival_rate_NA

In [ ]:
for df in [df_train, df_test]:
    df['Survival_Rate'] = (df['Ticket_Survival_Rate'] + df['Family_Survival_Rate']) / 2
    df['Survival_Rate_NA'] = (df['Ticket_Survival_Rate_NA'] + df['Family_Survival_Rate_NA']) / 2

### **2.5 Feature Transformation**

#### **2.5.1 Label Encoding Non-Numerical Features**
`Embarked`, `Sex`, `Deck` , `Title` and `Family_Size_Grouped` are object type, and `Age` and `Fare` features are category type. They are converted to numerical type with `LabelEncoder`. `LabelEncoder` basically labels the classes from **0** to **n**. This process is necessary for models to learn from those features.

In [ ]:
non_numeric_features = ['Embarked', 'Sex', 'Deck', 'Title', 'Family_Size_Grouped', 'Age', 'Fare']

for df in dfs:
    for feature in non_numeric_features:
        df[feature] = LabelEncoder().fit_transform(df[feature])

#### **2.5.2 One-Hot Encoding the Categorical Features**
The categorical features (`Pclass`, `Sex`, `Deck`, `Embarked`, `Title`) are converted to one-hot encoded features with `OneHotEncoder`. `Age` and `Fare` features are not converted because they are ordinal unlike the previous ones.

In [ ]:
cat_features = ['Pclass', 'Sex', 'Deck', 'Embarked', 'Title', 'Family_Size_Grouped']
encoded_features = []

for df in dfs:
    for feature in cat_features:
        encoded_feat = OneHotEncoder().fit_transform(df[feature].values.reshape(-1, 1)).toarray()
        n = df[feature].nunique()
        cols = ['{}_{}'.format(feature, n) for n in range(1, n + 1)]
        encoded_df = pd.DataFrame(encoded_feat, columns=cols)
        encoded_df.index = df.index
        encoded_features.append(encoded_df)

df_train = pd.concat([df_train, *encoded_features[:6]], axis=1)
df_test = pd.concat([df_test, *encoded_features[6:]], axis=1)

### **2.6 Conclusion**
`Age` and `Fare` features are binned. Binning helped dealing with outliers and it revealed some homogeneous groups in those features. `Family_Size` is created by adding `Parch` and `SibSp` features and **1**. `Ticket_Frequency` is created by counting the occurence of `Ticket` values.

`Name` feature is very useful. First, `Title` and `Is_Married` features are created from the title prefix in the names. Second, `Family_Survival_Rate` and `Family_Survival_Rate_NA`  features are created by target encoding the surname of the passengers. `Ticket_Survival_Rate` is created by target encoding the `Ticket` feature. `Survival_Rate` feature is created by averaging the `Family_Survival_Rate` and `Ticket_Survival_Rate` features.

Finally, the non-numeric type features are label encoded and categorical features are one-hot encoded. Created **5** new features (`Family_Size`, `Title`, `Is_Married`, `Survival_Rate` and `Survival_Rate_NA`) and dropped the useless features after encoding.

In [ ]:
df_all = concat_df(df_train, df_test)
drop_cols = ['Deck', 'Embarked', 'Family', 'Family_Size', 'Family_Size_Grouped', 'Survived',
             'Name', 'Parch', 'PassengerId', 'Pclass', 'Sex', 'SibSp', 'Ticket', 'Title',
            'Ticket_Survival_Rate', 'Family_Survival_Rate', 'Ticket_Survival_Rate_NA', 'Family_Survival_Rate_NA']

df_all.drop(columns=drop_cols, inplace=True)

df_all.head()

In [ ]:
# Redefining concat_df here to ensure it's available for the next cell, in case previous cells were not run.
def concat_df(train_data, test_data):
    # Returns a concatenated df of training and test set
    return pd.concat([train_data, test_data], sort=True).reset_index(drop=True)


In [ ]:
# Recreate df_all for clustering with all engineered features, before final column drops for model training.
# This ensures 'Survived' is still present (for training data rows) and all engineered features are available.
df_all_for_clustering = concat_df(df_train, df_test)

# Display the head to confirm df_all_for_clustering is correctly formed
print("df_all_for_clustering head:")
display(df_all_for_clustering.head())

In [ ]:
# Pilih fitur penting (sesuaikan dengan dataset kamu) untuk clustering
fitur_cluster = ['Age', 'Fare', 'Ticket_Frequency', 'Survival_Rate']

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_cluster = scaler.fit_transform(df_all_for_clustering[fitur_cluster])

from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=2, random_state=42) # Menggunakan 2 cluster sebagai contoh
df_all_for_clustering['target_cluster'] = kmeans.fit_predict(X_cluster)

# Lihat rata-rata tiap cluster untuk interpretasi
print("\nRata-rata fitur per cluster:")
print(df_all_for_clustering.groupby('target_cluster')[fitur_cluster].mean())

# Opsional: Lihat distribusi cluster dan kaitannya dengan Survived (jika Survived ada)
print("\nDistribusi 'target_cluster' terhadap 'Survived' (jika ada):")
if 'Survived' in df_all_for_clustering.columns:
    print(pd.crosstab(df_all_for_clustering['target_cluster'], df_all_for_clustering['Survived']))
else:
    print("'Survived' column not found in df_all_for_clustering for cross-tabulation.")

### **3.1.1 Elbow Method for Optimal K**

Untuk menentukan jumlah cluster ($k$) yang optimal, kita akan menggunakan metode Elbow. Metode ini menghitung *sum of squared distances* (inertia) untuk berbagai nilai $k$ dan memplotnya. Titik 'siku' pada grafik menunjukkan $k$ yang paling sesuai, di mana penambahan cluster lebih lanjut tidak memberikan penurunan inertia yang signifikan.

In [ ]:
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

# Calculate inertia for a range of k values
inertia = []
K_range = range(1, 11) # Test k from 1 to 10

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10) # n_init is set to 10 to suppress future warnings
    kmeans.fit(X_cluster)
    inertia.append(kmeans.inertia_)

# Plot the Elbow Method
plt.figure(figsize=(10, 6))
plt.plot(K_range, inertia, marker='o')
plt.xlabel('Number of Clusters (k)', size=15, labelpad=15)
plt.ylabel('Inertia', size=15, labelpad=15)
plt.title('Elbow Method for Optimal K', size=18, y=1.05)
plt.xticks(K_range)
plt.grid(True)
plt.show()

print("Grafik di atas menunjukkan metode Elbow. Titik 'siku' (elbow) pada grafik dapat membantu menentukan jumlah cluster (k) yang optimal.")

### **3.1.2 Applying KMeans Clustering**

Berdasarkan metode Elbow (misalnya, jika kita mengamati 'siku' di $k=3$ atau $k=4$, atau berdasarkan output clustering sebelumnya yang menggunakan 2 cluster), kita akan menerapkan KMeans dengan jumlah cluster yang dipilih dan menambahkan label cluster ke DataFrame.

In [ ]:
# Choose the optimal k based on the Elbow plot. For demonstration, let's use k=3 as an example.
optimal_k = 3

kmeans_optimal = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
df_all_for_clustering['cluster_labels'] = kmeans_optimal.fit_predict(X_cluster)

print(f"Distribusi cluster dengan k={optimal_k}:")
print(df_all_for_clustering['cluster_labels'].value_counts())

# Tampilkan rata-rata fitur per cluster untuk interpretasi
print("\nRata-rata fitur per cluster setelah clustering dengan k={optimal_k}:")
print(df_all_for_clustering.groupby('cluster_labels')[fitur_cluster].mean())

### **3.1.3 Visualisasi Cluster**

Untuk memvisualisasikan cluster, terutama ketika memiliki lebih dari dua fitur, kita dapat menggunakan Principal Component Analysis (PCA) untuk mengurangi dimensi data menjadi 2 komponen utama. Ini memungkinkan kita untuk memplot cluster dalam ruang 2D.

### **3.1.4 Integrating Cluster Labels as a Feature**

We will now integrate the `cluster_labels` obtained from KMeans as a new feature into our `df_train` and `df_test` dataframes. Since cluster labels are categorical, they will be one-hot encoded to avoid imposing an artificial ordinal relationship.

This new feature will then be included when preparing the `X_train` and `X_test` datasets for the classification models.

In [ ]:
# Extract cluster labels for training and test sets
train_cluster_labels = df_all_for_clustering.loc[df_train.index, 'cluster_labels']
test_cluster_labels = df_all_for_clustering.loc[df_test.index, 'cluster_labels']

# Add cluster labels as a new column 'Cluster' to df_train and df_test
df_train['Cluster'] = train_cluster_labels
df_test['Cluster'] = test_cluster_labels

# One-hot encode the new 'Cluster' feature
from sklearn.preprocessing import OneHotEncoder

# Concatenate train and test 'Cluster' columns for consistent encoding across both datasets
all_clusters = pd.concat([df_train['Cluster'], df_test['Cluster']], axis=0)

ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
cluster_encoded = ohe.fit_transform(all_clusters.values.reshape(-1, 1))

# Create DataFrame for encoded clusters
num_clusters = all_clusters.nunique()
cluster_cols = ['Cluster_{}'.format(i+1) for i in range(num_clusters)]
df_cluster_encoded = pd.DataFrame(cluster_encoded, columns=cluster_cols, index=all_clusters.index)

# Re-split into train and test encoded cluster features
df_train_cluster_encoded = df_cluster_encoded.loc[df_train.index]
df_test_cluster_encoded = df_cluster_encoded.loc[df_test.index]

# Concatenate back to df_train and df_test, dropping the original 'Cluster' column
df_train = pd.concat([df_train.drop(columns=['Cluster']), df_train_cluster_encoded], axis=1)
df_test = pd.concat([df_test.drop(columns=['Cluster']), df_test_cluster_encoded], axis=1)

print("df_train columns after adding cluster features:")
print(df_train.columns)
print("\ndf_test columns after adding cluster features:")
print(df_test.columns)


### **3.5 Re-preparing Data for Model Training**

With the new cluster features added, we need to redefine `X_train`, `y_train`, and `X_test` to include this information. The `drop_cols` list will be used to remove non-feature columns (like 'Survived', 'PassengerId', and original features that have been replaced by one-hot encoded versions) from the input `X` matrices.

In [ ]:
import pandas as pd

# Define the columns to drop from the feature set (X_train and X_test)
# This includes original string/identifier features, features replaced by one-hot encoding,
# intermediate features, and the old target variable 'y_cluster_logic'.
# 'Survived' is explicitly removed from X_train as it will now be the target.
features_to_drop_from_X = [
    'PassengerId', 'Name', 'Ticket', 'Family', # Identifiers/original strings
    'Deck', 'Embarked', 'Pclass', 'Sex', 'Title', 'Family_Size_Grouped', # Original categorical features (now OHE)
    'Family_Size', 'Parch', 'SibSp', # Combined into other features
    'Ticket_Survival_Rate', 'Family_Survival_Rate', 'Ticket_Survival_Rate_NA', 'Family_Survival_Rate_NA', # Combined into Survival_Rate
    'cluster_labels', 'y_cluster_logic', # Intermediate/old target, OHE versions of cluster_labels are kept
    'Survived' # This is now the target variable, so remove from features
]

# Ensure no duplicates and convert to a set for efficient lookup
features_to_drop_from_X = list(set(features_to_drop_from_X))

# Prepare X_train by dropping the specified columns
X_train = StandardScaler().fit_transform(df_train.drop(columns=features_to_drop_from_X, errors='ignore'))
# Set y_train to the actual 'Survived' target variable
y_train = df_train['Survived'].values

# Prepare X_test by dropping the specified columns
# Note: 'Survived' column does not exist in df_test, so errors='ignore' handles this gracefully.
X_test = StandardScaler().fit_transform(df_test.drop(columns=features_to_drop_from_X, errors='ignore'))

print('X_train shape with cluster features: {}'.format(X_train.shape))
print('y_train shape: {}'.format(y_train.shape))
print('X_test shape with cluster features: {}'.format(X_test.shape))

In [ ]:
from sklearn.decomposition import PCA
import seaborn as sns

# Apply PCA to reduce dimensions to 2 for visualization
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_cluster)

# Add PCA components to the DataFrame for plotting
df_all_for_clustering['PCA1'] = X_pca[:, 0]
df_all_for_clustering['PCA2'] = X_pca[:, 1]

# Plot the clusters
plt.figure(figsize=(12, 8))
sns.scatterplot(x='PCA1', y='PCA2', hue='cluster_labels', data=df_all_for_clustering,
                palette='viridis', s=100, alpha=0.8, legend='full')
plt.title(f'KMeans Clusters (k={optimal_k}) Visualized with PCA', size=18, y=1.05)
plt.xlabel('Principal Component 1', size=15, labelpad=15)
plt.ylabel('Principal Component 2', size=15, labelpad=15)
plt.grid(True)
plt.show()

# Optional: Visualize specific feature pairs against clusters
plt.figure(figsize=(15, 5))
plt.subplot(1, 2, 1)
sns.scatterplot(x='Age', y='Fare', hue='cluster_labels', data=df_all_for_clustering,
                palette='viridis', s=100, alpha=0.8, legend='full')
plt.title(f'Clusters by Age and Fare (k={optimal_k})', size=14)

plt.subplot(1, 2, 2)
sns.scatterplot(x='Ticket_Frequency', y='Survival_Rate', hue='cluster_labels', data=df_all_for_clustering,
                palette='viridis', s=100, alpha=0.8, legend='full')
plt.title(f'Clusters by Ticket Frequency and Survival Rate (k={optimal_k})', size=14)
plt.tight_layout()
plt.show()

## **3. Model**

In [ ]:
# =========================
# RANDOM FOREST MODEL
# =========================

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Model 1: Single model
single_best_model = RandomForestClassifier(random_state=42)
single_best_model.fit(X_train, y_train)

y_pred_single = single_best_model.predict(X_test)
# acc_single = accuracy_score(y_test, y_pred_single) # y_test is not available for the actual test set

print("Predictions for Single Model generated.")


# Model 2: Model kedua (misalnya parameter berbeda)
leaderboard_model = RandomForestClassifier(n_estimators=100, random_state=42)
leaderboard_model.fit(X_train, y_train)

y_pred_leader = leaderboard_model.predict(X_test)
# acc_leader = accuracy_score(y_test, y_pred_leader) # y_test is not available for the actual test set

print("Predictions for Leaderboard Model generated.")


In [ ]:
# ===============================
# 3.1 RANDOM FOREST
# ===============================

"""
Created 2 RandomForestClassifier:
- Single model
- Model untuk k-fold cross validation

Single best model:
Accuracy sekitar 0.82775 (leaderboard)
Cocok untuk eksperimen awal dan tuning

Leaderboard model:
Accuracy sekitar 0.83732 (5-fold CV)
Cenderung overfitting (dibuat untuk skor kompetisi)

Kesimpulan:
- Leaderboard model tidak disarankan untuk real case
- Single best model lebih stabil untuk pembelajaran
"""

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# ===============================
# Model 1: Single Best Model
# ===============================
single_best_model = RandomForestClassifier(random_state=42)
single_best_model.fit(X_train, y_train)

y_pred_single = single_best_model.predict(X_test)
# acc_single = accuracy_score(y_test, y_pred_single) # y_test is not available for the actual test set

print("Predictions for Single Model generated.")


# ===============================
# Model 2: Leaderboard Model
# ===============================
leaderboard_model = RandomForestClassifier(n_estimators=100, random_state=42)
leaderboard_model.fit(X_train, y_train)

y_pred_leader = leaderboard_model.predict(X_test)
# acc_leader = accuracy_score(y_test, y_pred_leader) # y_test is not available for the actual test set

print("Predictions for Leaderboard Model generated.")

Dalam sel-sel yang disediakan, dua model RandomForestClassifier yaitu single_best_model dan leaderboard_model telah diinisialisasi dan dilatih menggunakan fitur pelatihan (X_train) dan variabel target 'Survived' (y_train) dari data yang sudah diproses. single_best_model berfungsi sebagai model dasar untuk eksperimen awal, sementara leaderboard_model dirancang untuk mencapai skor kompetisi yang tinggi, yang terkadang berarti overfitting ringan. Kedua model berhasil membuat prediksi (y_pred_single dan y_pred_leader) pada set data uji (X_test), meskipun y_test (label sebenarnya untuk set data uji) tidak tersedia, yang merupakan praktik umum dalam kompetisi machine learning. Dengan demikian, model-model tersebut berhasil dilatih dan menghasilkan prediksi, dengan evaluasi lebih lanjut terhadap kinerja (seperti akurasi, presisi, recall, dan matriks kebingungan) yang dibahas pada bagian evaluasi model di notebook.

# This cell is a duplicate and has been removed.

In [ ]:
single_best_model = RandomForestClassifier(criterion='gini',
                                           n_estimators=1100,
                                           max_depth=5,
                                           min_samples_split=4,
                                           min_samples_leaf=5,
                                           max_features='sqrt',
                                           oob_score=True,
                                           random_state=SEED,
                                           n_jobs=-1,
                                           verbose=1)

leaderboard_model = RandomForestClassifier(criterion='gini',
                                           n_estimators=1750,
                                           max_depth=7,
                                           min_samples_split=6,
                                           min_samples_leaf=6,
                                           max_features='sqrt',
                                           oob_score=True,
                                           random_state=SEED,
                                           n_jobs=-1,
                                           verbose=1)

print("Random Forest models initialized. They will be trained using the updated X_train and y_train.")

### **3.2 CLASSIFICATION Feature Importance**

## Tujuan Klasifikasi

Tujuan utama dari bagian klasifikasi dalam notebook ini adalah untuk memprediksi apakah seorang penumpang selamat (`Survived = 1`) atau tidak selamat (`Survived = 0`) dari bencana Titanic. Ini merupakan masalah klasifikasi biner, di mana model akan dilatih untuk mengidentifikasi pola dalam fitur-fitur penumpang yang berkorelasi dengan tingkat kelangsungan hidup.

### Metrik Evaluasi:
Untuk mengukur kinerja model klasifikasi, kami menggunakan beberapa metrik penting:

*   **Accuracy (Akurasi):** Proporsi prediksi yang benar dari total prediksi. Ini adalah metrik umum untuk memberikan gambaran keseluruhan seberapa baik model berkinerja.
*   **Precision (Presisi):** Proporsi positif sejati dari semua hasil positif yang diprediksi. Ini penting ketika biaya False Positives tinggi (misalnya, memprediksi seseorang selamat padahal tidak).
*   **Recall (Sensitivitas/Tingkat Penemuan):** Proporsi positif sejati yang diidentifikasi dengan benar dari semua aktual positif. Ini penting ketika biaya False Negatives tinggi (misalnya, memprediksi seseorang tidak selamat padahal sebenarnya selamat).
*   **F1-Score:** Rata-rata harmonik dari Precision dan Recall. Metrik ini memberikan keseimbangan antara Precision dan Recall, yang berguna ketika distribusi kelas tidak seimbang.
*   **Confusion Matrix:** Sebuah tabel yang menunjukkan jumlah True Positives, True Negatives, False Positives, dan False Negatives, memberikan gambaran detail tentang jenis kesalahan yang dibuat oleh model.
*   **ROC Curve (Receiver Operating Characteristic Curve) & AUC (Area Under the Curve):** ROC Curve memvisualisasikan kemampuan model dalam membedakan antara kelas positif dan negatif pada berbagai ambang batas klasifikasi. AUC mengukur area di bawah kurva ROC; nilai AUC yang lebih tinggi menunjukkan kinerja model yang lebih baik dalam membedakan kelas.

### **3.2 Decision Tree Classifier**

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

dt_model = DecisionTreeClassifier(random_state=SEED)
dt_model.fit(X_train, y_train)
y_pred_dt = dt_model.predict(X_test)

print(f"Decision Tree Classifier Training Accuracy: {accuracy_score(y_train, dt_model.predict(X_train)):.4f}")

### **3.3 K-Nearest Neighbors (KNN) Classifier**

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

knn_model = KNeighborsClassifier(n_neighbors=5) # Using 5 neighbors as a common starting point
knn_model.fit(X_train, y_train)
y_pred_knn = knn_model.predict(X_test)

print(f"KNN Classifier Training Accuracy: {accuracy_score(y_train, knn_model.predict(X_train)):.4f}")

### **3.4 Gaussian Naive Bayes Classifier**

In [ ]:
from sklearn.naive_bayes import GaussianNB

nb_model = GaussianNB()
nb_model.fit(X_train, y_train)
y_pred_nb = nb_model.predict(X_test)

print(f"Naive Bayes Classifier Training Accuracy: {accuracy_score(y_train, nb_model.predict(X_train)):.4f}")

### **3.5 Logistic Regression**

In [ ]:
from sklearn.linear_model import LogisticRegression

log_reg_model = LogisticRegression(random_state=SEED, solver='liblinear') # 'liblinear' is good for small datasets
log_reg_model.fit(X_train, y_train)
y_pred_lr = log_reg_model.predict(X_test)

print(f"Logistic Regression Training Accuracy: {accuracy_score(y_train, log_reg_model.predict(X_train)):.4f}")

### **3.6 Support Vector Machine (SVM)**

In [ ]:
from sklearn.svm import SVC

svm_model = SVC(random_state=SEED, probability=True) # probability=True to allow .predict_proba later if needed
svm_model.fit(X_train, y_train)
y_pred_svm = svm_model.predict(X_test)

print(f"SVM Classifier Training Accuracy: {accuracy_score(y_train, svm_model.predict(X_train)):.4f}")

### **3.7 Model Evaluation on Test Data**

In [ ]:
from sklearn.metrics import accuracy_score

print("--- Model Training Accuracies ---")
# Decision Tree Classifier
print(f"Decision Tree Classifier Training Accuracy: {accuracy_score(y_train, dt_model.predict(X_train)):.4f}")

# K-Nearest Neighbors (KNN) Classifier
print(f"KNN Classifier Training Accuracy: {accuracy_score(y_train, knn_model.predict(X_train)):.4f}")

# Gaussian Naive Bayes Classifier
print(f"Naive Bayes Classifier Training Accuracy: {accuracy_score(y_train, nb_model.predict(X_train)):.4f}")

# Logistic Regression
print(f"Logistic Regression Training Accuracy: {accuracy_score(y_train, log_reg_model.predict(X_train)):.4f}")

# Support Vector Machine (SVM)
print(f"SVM Classifier Training Accuracy: {accuracy_score(y_train, svm_model.predict(X_train)):.4f}")

print("\n--- Predictions on Test Data (X_test) ---")
print("Predictions for Decision Tree (first 5): ", y_pred_dt[:5])
print("Predictions for KNN (first 5): ", y_pred_knn[:5])
print("Predictions for Naive Bayes (first 5): ", y_pred_nb[:5])
print("Predictions for Logistic Regression (first 5): ", y_pred_lr[:5])
print("Predictions for SVM (first 5): ", y_pred_svm[:5])

# Note: Test set accuracy cannot be calculated without ground truth labels (y_test).
# The models have made predictions on X_test, which can be used for submission if required.

### **3.8 Classification Model Evaluation**

Since we do not have the ground truth labels (`y_test`) for the actual test set, we will evaluate the classification models based on their performance on the *training data* (`X_train`, `y_train`). This will show how well each model has learned the patterns in the data it was trained on.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

classification_models = {
    "Decision Tree": dt_model,
    "KNN": knn_model,
    "Naive Bayes": nb_model,
    "Logistic Regression": log_reg_model,
    "SVM": svm_model
}

# Prepare a list to store training accuracies for comparison
training_accuracies = []

print("--- Classification Model Training Evaluation ---")

for name, model in classification_models.items():
    print(f"\nEvaluating {name}:")
    y_train_pred = model.predict(X_train)

    # Accuracy
    acc = accuracy_score(y_train, y_train_pred)
    training_accuracies.append({'Model': name, 'Accuracy': acc})
    print(f"  Training Accuracy: {acc:.4f}")

    # Classification Report
    print("  Classification Report:")
    print(classification_report(y_train, y_train_pred))

    # Confusion Matrix
    cm = confusion_matrix(y_train, y_train_pred)
    plt.figure(figsize=(4, 3))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(f'Confusion Matrix - {name} (Training Data)')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.show()

# Convert to DataFrame for easier plotting
df_training_accuracies = pd.DataFrame(training_accuracies)

print("\n--- Comparison of Classification Models (Training Accuracy) ---")
plt.figure(figsize=(10, 6))
sns.barplot(x='Model', y='Accuracy', data=df_training_accuracies, palette='viridis')
plt.title('Classification Model Training Accuracy Comparison', size=16)
plt.xlabel('Model', size=12)
plt.ylabel('Accuracy', size=12)
plt.ylim(0.7, 1.0) # Set y-limit to better show differences
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()


### **Ringkasan Evaluasi Model Klasifikasi**

Berikut adalah ringkasan kinerja model-model klasifikasi pada data pelatihan (`X_train`, `y_train`), mencakup akurasi, presisi, recall, dan f1-score, serta matriks kebingungan untuk setiap model.

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

classification_models = {
    "Decision Tree": dt_model,
    "KNN": knn_model,
    "Naive Bayes": nb_model,
    "Logistic Regression": log_reg_model,
    "SVM": svm_model
}

# Prepare lists to store metrics
model_names = []
accuracies = []
precisions_0 = []
recalls_0 = []
f1_scores_0 = []
precisions_1 = []
recalls_1 = []
f1_scores_1 = []

print("--- Ringkasan Metrik Model Klasifikasi (Data Pelatihan) ---")

for name, model in classification_models.items():
    y_train_pred = model.predict(X_train)

    model_names.append(name)
    accuracies.append(accuracy_score(y_train, y_train_pred))

    # Classification Report to get precision, recall, f1-score for each class
    report = classification_report(y_train, y_train_pred, output_dict=True)

    precisions_0.append(report['0.0']['precision'])
    recalls_0.append(report['0.0']['recall'])
    f1_scores_0.append(report['0.0']['f1-score'])

    precisions_1.append(report['1.0']['precision'])
    recalls_1.append(report['1.0']['recall'])
    f1_scores_1.append(report['1.0']['f1-score'])

    # Confusion Matrix
    cm = confusion_matrix(y_train, y_train_pred)
    plt.figure(figsize=(4, 3))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(f'Confusion Matrix - {name} (Training Data)')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.show()

# Create a DataFrame for the summary table
summary_df = pd.DataFrame({
    'Model': model_names,
    'Accuracy': accuracies,
    'Precision (Class 0)': precisions_0,
    'Recall (Class 0)': recalls_0,
    'F1-Score (Class 0)': f1_scores_0,
    'Precision (Class 1)': precisions_1,
    'Recall (Class 1)': recalls_1,
    'F1-Score (Class 1)': f1_scores_1
})

print("\n--- Tabel Ringkasan Metrik Klasifikasi ---")
# Set display options to avoid truncation
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
display(summary_df.sort_values(by='Accuracy', ascending=False))
# Reset display options to default after showing the table
pd.reset_option('display.max_rows')
pd.reset_option('display.max_columns')

### **4. Regression Models**

Kita akan menerapkan beberapa model regresi untuk memprediksi `y_train` (`y_cluster_logic`). Karena `y_train` adalah biner, model regresi akan memprediksi skor kontinu yang dapat diinterpretasikan sebagai probabilitas.

#### **4.1 Split Data untuk Evaluasi Regresi**

Sebelum melatih model, kita akan membagi `X_train` dan `y_train` menjadi subset pelatihan dan validasi untuk mengevaluasi kinerja model secara lokal.

In [ ]:
from sklearn.model_selection import train_test_split

# Membagi X_train dan y_train yang sudah ada menjadi subset pelatihan dan validasi
X_train_sub, X_val_sub, y_train_sub, y_val_sub = train_test_split(X_train, y_train, test_size=0.2, random_state=SEED, stratify=y_train)

print(f"X_train_sub shape: {X_train_sub.shape}")
print(f"y_train_sub shape: {y_train_sub.shape}")
print(f"X_val_sub shape: {X_val_sub.shape}")
print(f"y_val_sub shape: {y_val_sub.shape}")

#### **4.2 Linear Regression**

Model regresi linier sederhana untuk memprediksi target kontinu.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

# Inisialisasi dan latih model Linear Regression
lin_reg_model = LinearRegression()
lin_reg_model.fit(X_train_sub, y_train_sub)

# Prediksi pada set validasi
y_pred_lin_reg = lin_reg_model.predict(X_val_sub)

# Evaluasi model
mae_lin_reg = mean_absolute_error(y_val_sub, y_pred_lin_reg)
mse_lin_reg = mean_squared_error(y_val_sub, y_pred_lin_reg)
rmse_lin_reg = np.sqrt(mse_lin_reg)
r2_lin_reg = r2_score(y_val_sub, y_pred_lin_reg)

print("--- Linear Regression Evaluation ---")
print(f"Mean Absolute Error (MAE): {mae_lin_reg:.4f}")
print(f"Mean Squared Error (MSE): {mse_lin_reg:.4f}")
print(f"Root Mean Squared Error (RMSE): {rmse_lin_reg:.4f}")
print(f"R-squared (R2): {r2_lin_reg:.4f}")

#### **4.3 Decision Tree Regression**

Model regresi pohon keputusan, yang dapat menangkap hubungan non-linier dalam data.

In [ ]:
from sklearn.tree import DecisionTreeRegressor

# Inisialisasi dan latih model Decision Tree Regressor
dt_reg_model = DecisionTreeRegressor(random_state=SEED)
dt_reg_model.fit(X_train_sub, y_train_sub)

# Prediksi pada set validasi
y_pred_dt_reg = dt_reg_model.predict(X_val_sub)

# Evaluasi model
mae_dt_reg = mean_absolute_error(y_val_sub, y_pred_dt_reg)
mse_dt_reg = mean_squared_error(y_val_sub, y_pred_dt_reg)
rmse_dt_reg = np.sqrt(mse_dt_reg)
r2_dt_reg = r2_score(y_val_sub, y_pred_dt_reg)

print("--- Decision Tree Regression Evaluation ---")
print(f"Mean Absolute Error (MAE): {mae_dt_reg:.4f}")
print(f"Mean Squared Error (MSE): {mse_dt_reg:.4f}")
print(f"Root Mean Squared Error (RMSE): {rmse_dt_reg:.4f}")
print(f"R-squared (R2): {r2_dt_reg:.4f}")

#### **4.4 Random Forest Regression**

Ensemble model yang menggunakan banyak pohon keputusan untuk meningkatkan akurasi dan mengurangi overfitting.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

# Inisialisasi dan latih model Random Forest Regressor
rf_reg_model = RandomForestRegressor(random_state=SEED, n_estimators=100, n_jobs=-1)
rf_reg_model.fit(X_train_sub, y_train_sub)

# Prediksi pada set validasi
y_pred_rf_reg = rf_reg_model.predict(X_val_sub)

# Evaluasi model
mae_rf_reg = mean_absolute_error(y_val_sub, y_pred_rf_reg)
mse_rf_reg = mean_squared_error(y_val_sub, y_pred_rf_reg)
rmse_rf_reg = np.sqrt(mse_rf_reg)
r2_rf_reg = r2_score(y_val_sub, y_pred_rf_reg)

print("--- Random Forest Regression Evaluation ---")
print(f"Mean Absolute Error (MAE): {mae_rf_reg:.4f}")
print(f"Mean Squared Error (MSE): {mse_rf_reg:.4f}")
print(f"Root Mean Squared Error (RMSE): {rmse_rf_reg:.4f}")
print(f"R-squared (R2): {r2_rf_reg:.4f}")

#### **4.5 Ringkasan Hasil Regresi**

Berikut adalah ringkasan kinerja model regresi yang telah diuji:

In [ ]:
results = {
    "Model": ["Linear Regression", "Decision Tree Regression", "Random Forest Regression"],
    "MAE": [mae_lin_reg, mae_dt_reg, mae_rf_reg],
    "MSE": [mse_lin_reg, mse_dt_reg, mse_rf_reg],
    "RMSE": [rmse_lin_reg, rmse_dt_reg, rmse_rf_reg],
    "R2": [r2_lin_reg, r2_dt_reg, r2_rf_reg]
}

df_results = pd.DataFrame(results)
display(df_results.sort_values(by='RMSE'))

### **4.6 Regression Model Evaluation Summary and Comparison**

Here, we will recap the performance metrics for the regression models on the validation set (`X_val_sub`, `y_val_sub`) and visualize their comparison.

In [ ]:
print("--- Regression Model Evaluation Summary ---")
display(df_results.sort_values(by='RMSE'))

# Visualize Regression Model Comparison (e.g., by RMSE and R2 Score)
fig, axs = plt.subplots(1, 2, figsize=(16, 6))

sns.barplot(x='Model', y='RMSE', data=df_results.sort_values(by='RMSE'), palette='magma', ax=axs[0])
axs[0].set_title('Regression Model RMSE Comparison', size=16)
axs[0].set_xlabel('Model', size=12)
axs[0].set_ylabel('RMSE', size=12)
axs[0].grid(axis='y', linestyle='--', alpha=0.7)

sns.barplot(x='Model', y='R2', data=df_results.sort_values(by='R2', ascending=False), palette='viridis', ax=axs[1])
axs[1].set_title('Regression Model R-squared (R2) Comparison', size=16)
axs[1].set_xlabel('Model', size=12)
axs[1].set_ylabel('R2 Score', size=12)
axs[1].set_ylim(0.999, 1.001) # Adjust y-limit for better visualization of high R2 scores
axs[1].grid(axis='y', linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()


`StratifiedKFold` is used for stratifying the target variable. The folds are made by preserving the percentage of samples for each class in target variable (`Survived`).

In [ ]:
N = 5
oob = 0
probs = pd.DataFrame(np.zeros((len(X_test), N * 2)), columns=['Fold_{}_Prob_{}'.format(i, j) for i in range(1, N + 1) for j in range(2)])
importances = pd.DataFrame(np.zeros((X_train.shape[1], N)), columns=['Fold_{}'.format(i) for i in range(1, N + 1)], index=df_train.drop(columns=features_to_drop_from_X, errors='ignore').columns)
fprs, tprs, scores = [], [], []

skf = StratifiedKFold(n_splits=N, random_state=N, shuffle=True)

for fold, (trn_idx, val_idx) in enumerate(skf.split(X_train, y_train), 1):
    print('Fold {}'.format(fold))

    # Fitting the model
    leaderboard_model.fit(X_train[trn_idx], y_train[trn_idx])

    # Computing Train AUC score
    trn_fpr, trn_tpr, trn_thresholds = roc_curve(y_train[trn_idx], leaderboard_model.predict_proba(X_train[trn_idx])[:, 1])
    trn_auc_score = auc(trn_fpr, trn_tpr)
    # Computing Validation AUC score
    val_fpr, val_tpr, val_thresholds = roc_curve(y_train[val_idx], leaderboard_model.predict_proba(X_train[val_idx])[:, 1])
    val_auc_score = auc(val_fpr, val_tpr)

    scores.append((trn_auc_score, val_auc_score))
    fprs.append(val_fpr)
    tprs.append(val_tpr)

    # X_test probabilities
    probs.loc[:, 'Fold_{}_Prob_0'.format(fold)] = leaderboard_model.predict_proba(X_test)[:, 0]
    probs.loc[:, 'Fold_{}_Prob_1'.format(fold)] = leaderboard_model.predict_proba(X_test)[:, 1]
    importances.iloc[:, fold - 1] = leaderboard_model.feature_importances_

    oob += leaderboard_model.oob_score_ / N
    print('Fold {} OOB Score: {}'.format(fold, leaderboard_model.oob_score_))

print('Average OOB Score: {}'.format(oob))

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=-1)]: Done  46 tasks      | elapsed:    0.3s
[Parallel(n_jobs=-1)]: Done 196 tasks      | elapsed:    1.0s
[Parallel(n_jobs=-1)]: Done 446 tasks      | elapsed:    2.1s
[Parallel(n_jobs=-1)]: Done 796 tasks      | elapsed:    3.2s
[Parallel(n_jobs=-1)]: Done 1246 tasks      | elapsed:    4.4s
[Parallel(n_jobs=-1)]: Done 1750 out of 1750 | elapsed:    5.4s finished
[Parallel(n_jobs=2)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done  46 tasks      | elapsed:    0.0s
[Parallel(n_jobs=2)]: Done 196 tasks      | elapsed:    0.1s
[Parallel(n_jobs=2)]: Done 446 tasks      | elapsed:    0.1s
[Parallel(n_jobs=2)]: Done 796 tasks      | elapsed:    0.2s
[Parallel(n_jobs=2)]: Done 1246 tasks      | elapsed:    0.4s
[Parallel(n_jobs=2)]: Done 1750 out of 1750 | elapsed:    0.6s finished
[Parallel(n_jobs=2)]: Using backend ThreadingBackend with 2 concurrent worker

Fold 2 OOB Score: 0.8359046283309958
Fold 3


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=-1)]: Done  46 tasks      | elapsed:    0.1s
[Parallel(n_jobs=-1)]: Done 196 tasks      | elapsed:    0.4s
[Parallel(n_jobs=-1)]: Done 446 tasks      | elapsed:    0.9s
[Parallel(n_jobs=-1)]: Done 796 tasks      | elapsed:    1.7s
[Parallel(n_jobs=-1)]: Done 1246 tasks      | elapsed:    2.6s
[Parallel(n_jobs=-1)]: Done 1750 out of 1750 | elapsed:    3.8s finished


Insight dari Model Klasifikasi (Dievaluasi pada Data Pelatihan)
Dari evaluasi model klasifikasi pada data pelatihan, kita bisa melihat perbedaan kinerja yang signifikan antar model:

*   **Decision Tree:** Model ini menunjukkan akurasi pelatihan tertinggi (sekitar 0.9529). Ini berarti Decision Tree sangat baik dalam mempelajari pola pada data pelatihan. Namun, akurasi yang sangat tinggi pada data pelatihan seringkali menjadi indikasi overfitting, di mana model terlalu spesifik pada data yang dilihatnya dan mungkin tidak akan berkinerja sebaik itu pada data baru yang belum pernah dilihat. Precision, Recall, dan F1-score untuk kedua kelas juga sangat tinggi, mengonfirmasi kemampuan model dalam membedakan kelas pada data latih.
*   **SVM:** Model ini menempati posisi kedua dengan akurasi pelatihan 0.8799. SVM menunjukkan kinerja yang solid dengan keseimbangan yang baik antara precision dan recall untuk kedua kelas, menjadikannya pilihan yang cukup handal.
*   **KNN dan Logistic Regression:** Kedua model ini memiliki akurasi yang mirip, yaitu 0.8653 untuk KNN dan 0.8631 untuk Logistic Regression. Keduanya cenderung lebih baik dalam memprediksi kelas 'Not Survived' (kelas 0) dibandingkan 'Survived' (kelas 1), yang terlihat dari nilai recall kelas 0 yang lebih tinggi. Ini mungkin mengindikasikan bahwa data kelas 0 lebih mudah diidentifikasi.
*   **Gaussian Naive Bayes:** Model ini menunjukkan akurasi pelatihan terendah (0.8272). Meskipun demikian, model ini masih memberikan kinerja yang layak dan bisa menjadi baseline. Precision untuk kelas 1 (Survived) adalah yang terendah di antara model lain, menunjukkan bahwa model ini cenderung kurang akurat dalam mengidentifikasi penumpang yang selamat.

**Kesimpulan Model Klasifikasi:** Sementara Decision Tree unggul dalam 'menghafal' data pelatihan, kinerja sebenarnya pada data tak terlihat harus diperiksa lebih lanjut untuk menilai generalisasi. Model seperti SVM, KNN, dan Logistic Regression menawarkan kinerja yang lebih stabil dan cenderung lebih baik dalam generalisasi, meskipun dengan akurasi pelatihan yang sedikit lebih rendah.

Insight dari Model Regresi (Dievaluasi pada Data Validasi)
Model regresi digunakan untuk memprediksi `y_cluster_logic`, yang merupakan representasi probabilitas survival berdasarkan cluster. Evaluasi pada data validasi menghasilkan:

*   **Linear Regression:** Menunjukkan kinerja yang hampir sempurna dengan MAE, MSE, dan RMSE mendekati nol, serta nilai R-squared (R2) sebesar 1.0000. Ini mengindikasikan bahwa model linier dapat menjelaskan variansi dalam y_cluster_logic dengan sangat baik pada data validasi.
*   **Decision Tree Regression:** Juga menunjukkan kinerja yang sempurna dengan MAE, MSE, RMSE, dan R2 yang identik dengan Linear Regression. Ini tidak mengherankan karena Decision Tree dapat membagi data dengan sangat tepat hingga menghasilkan error nol pada set data yang sudah diproses secara deterministik (seperti `y_cluster_logic`).
*   **Random Forest Regression:** Model ini juga menunjukkan kinerja yang sangat kuat dengan MAE dan MSE yang sangat rendah, serta R2 yang mendekati 1.0000 (0.999951). Sedikit deviasi dari nol pada MAE, MSE, dan RMSE dibandingkan dengan Linear Regression dan Decision Tree Regression mungkin disebabkan oleh sifat ensemble dan acak dari Random Forest, tetapi secara keseluruhan, kinerja tetap luar biasa dalam memprediksi target biner yang didapat dari pengelompokan.

**Kesimpulan Model Regresi:** Hasil untuk model regresi sangat mengesankan, menunjukkan bahwa `y_cluster_logic` dapat diprediksi dengan akurasi yang hampir sempurna menggunakan fitur-fitur yang ada. Ini menggarisbawahi bagaimana representasi target yang berasal dari pengelompokan dapat diserap dengan baik oleh model-model ini. Namun, penting untuk dicatat bahwa kesempurnaan ini mungkin sebagian karena `y_cluster_logic` itu sendiri adalah target yang 'direkayasa' dari fitur yang sama, yang dapat mempermudah model untuk mempelajarinya.

### **3.2 Feature Importance**

In [ ]:
importances['Mean_Importance'] = importances.mean(axis=1)
importances.sort_values(by='Mean_Importance', inplace=True, ascending=False)

plt.figure(figsize=(15, 20))
sns.barplot(x='Mean_Importance', y=importances.index, data=importances)

plt.xlabel('')
plt.tick_params(axis='x', labelsize=15)
plt.tick_params(axis='y', labelsize=15)
plt.title('Random Forest Classifier Mean Feature Importance Between Folds', size=15)

plt.show()

### **Visualisasi Feature Importance**

Berikut adalah plot yang menunjukkan pentingnya fitur (feature importance) rata-rata dari model Random Forest (`leaderboard_model`) di setiap fold cross-validation. Ini membantu kita memahami fitur mana yang paling berpengaruh dalam prediksi survival.

In [ ]:
importances['Mean_Importance'] = importances.mean(axis=1)
importances.sort_values(by='Mean_Importance', inplace=True, ascending=False)

plt.figure(figsize=(15, 20))
sns.barplot(x='Mean_Importance', y=importances.index, data=importances)

plt.xlabel('')
plt.tick_params(axis='x', labelsize=15)
plt.tick_params(axis='y', labelsize=15)
plt.title('Random Forest Classifier Mean Feature Importance Between Folds', size=15)

plt.show()

### **3.3 ROC Curve**

### **Visualisasi ROC Curve**

Kurva ROC (Receiver Operating Characteristic) dan AUC (Area Under the Curve) memberikan gambaran tentang kinerja model dalam membedakan antara kelas positif (Survived) dan negatif (Not Survived) di setiap fold cross-validation. Semakin tinggi nilai AUC, semakin baik kinerja model.

In [ ]:
def plot_roc_curve(fprs, tprs):

    tprs_interp = []
    aucs = []
    mean_fpr = np.linspace(0, 1, 100)
    f, ax = plt.subplots(figsize=(15, 15))

    # Plotting ROC for each fold and computing AUC scores
    for i, (fpr, tpr) in enumerate(zip(fprs, tprs), 1):
        tprs_interp.append(np.interp(mean_fpr, fpr, tpr))
        tprs_interp[-1][0] = 0.0
        roc_auc = auc(fpr, tpr)
        aucs.append(roc_auc)
        ax.plot(fpr, tpr, lw=1, alpha=0.3, label='ROC Fold {} (AUC = {:.3f})'.format(i, roc_auc))

    # Plotting ROC for random guessing
    plt.plot([0, 1], [0, 1], linestyle='--', lw=2, color='r', alpha=0.8, label='Random Guessing')

    mean_tpr = np.mean(tprs_interp, axis=0)
    mean_tpr[-1] = 1.0
    mean_auc = auc(mean_fpr, mean_tpr)
    std_auc = np.std(aucs)

    # Plotting the mean ROC
    ax.plot(mean_fpr, mean_tpr, color='b', label='Mean ROC (AUC = {:.3f} $\pm$ {:.3f})'.format(mean_auc, std_auc), lw=2, alpha=0.8)

    # Plotting the standard deviation around the mean ROC Curve
    std_tpr = np.std(tprs_interp, axis=0)
    tprs_upper = np.minimum(mean_tpr + std_tpr, 1)
    tprs_lower = np.maximum(mean_tpr - std_tpr, 0)
    ax.fill_between(mean_fpr, tprs_lower, tprs_upper, color='grey', alpha=.2, label='$\pm$ 1 std. dev.')

    ax.set_xlabel('False Positive Rate', size=15, labelpad=20)
    ax.set_ylabel('True Positive Rate', size=15, labelpad=20)
    ax.tick_params(axis='x', labelsize=15)
    ax.tick_params(axis='y', labelsize=15)
    ax.set_xlim([-0.05, 1.05])
    ax.set_ylim([-0.05, 1.05])

    ax.set_title('ROC Curves of Folds', size=20, y=1.02)
    ax.legend(loc='lower right', prop={'size': 13})

    plt.show()

plot_roc_curve(fprs, tprs)

### **3.4 Submission**

In [ ]:
class_survived = [col for col in probs.columns if col.endswith('Prob_1')]
probs['1'] = probs[class_survived].sum(axis=1) / N
probs['0'] = probs.drop(columns=class_survived).sum(axis=1) / N
probs['pred'] = 0
pos = probs[probs['1'] >= 0.5].index
probs.loc[pos, 'pred'] = 1

y_pred = probs['pred'].astype(int)

submission_df = pd.DataFrame(columns=['PassengerId', 'Survived'])
submission_df['PassengerId'] = df_test['PassengerId']
submission_df['Survived'] = y_pred.values
submission_df.to_csv('submissions.csv', header=True, index=False)
submission_df.head(10)

In [ ]:
# Pilih fitur penting (sesuaikan dengan dataset kamu)
# Menggunakan fitur numerik yang relevan dari dataset Titanic yang sudah diproses
fitur_cluster = ['Age', 'Fare', 'Ticket_Frequency', 'Survival_Rate']

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
# Pastikan df_all sudah didefinisikan dari sel-sel sebelumnya
X_cluster = scaler.fit_transform(df_all[fitur_cluster])

from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=2, random_state=42) # Menggunakan 2 cluster sebagai contoh
df_all['target'] = kmeans.fit_predict(X_cluster)

# Lihat rata-rata tiap cluster untuk interpretasi
print(df_all.groupby('target')[fitur_cluster].mean())

In [ ]:
display(df_train.describe())

**NILAI** **Y**

In [ ]:
# Re-add 'cluster_labels' to df_train and df_test temporarily for this calculation
# The series train_cluster_labels and test_cluster_labels were created in cell c0a6b0a8
df_train['cluster_labels'] = train_cluster_labels
df_test['cluster_labels'] = test_cluster_labels

# 1. Calculate the actual mean 'Survived' rate for each cluster within the training data
# This uses the 'Survived' column which is only available in df_train
cluster_actual_survival_means = df_train.groupby('cluster_labels')['Survived'].mean()

# 2. Define a threshold to convert mean survival rate into a binary label
# If a cluster's mean survival rate is > 0.5, it's considered a 'high_survival_cluster' (1),
# otherwise, a 'low_survival_cluster' (0). This is the 'logika nilai' (value logic).
threshold = 0.5
cluster_binary_survival_propensity = (cluster_actual_survival_means > threshold).astype(int)

# 3. Create the new Y for training data (`y_cluster_logic`)
# Map the 'cluster_labels' in df_train to their binary survival propensity
df_train['y_cluster_logic'] = df_train['cluster_labels'].map(cluster_binary_survival_propensity)

# 4. Create a similar 'y' for test data (`y_cluster_logic`)
# This would serve as a cluster-based prediction if df_test didn't have 'Survived' (which it doesn't)
# Or as a feature if you want to use it in a meta-model.
df_test['y_cluster_logic'] = df_test['cluster_labels'].map(cluster_binary_survival_propensity)

print("Mean Survived rate per cluster (from training data):")
print(cluster_actual_survival_means)

print("\nBinary survival propensity per cluster (used for new Y value):")
print(cluster_binary_survival_propensity)

print("\nHead of df_train with original 'Survived', 'cluster_labels', and new 'y_cluster_logic':")
display(df_train[['Survived', 'cluster_labels', 'y_cluster_logic']].head())

### **Penjelasan Nilai Y (`y_cluster_logic`)**

Dalam konteks notebook ini, 'nilai Y' yang dimaksud adalah `y_cluster_logic`. Ini adalah variabel target biner baru yang direkayasa berdasarkan hasil dari proses clustering (pengelompokan data).

**Bagaimana `y_cluster_logic` Dibuat:**

1.  **Clustering:** Fitur-fitur penting (`Age`, `Fare`, `Ticket_Frequency`, `Survival_Rate`) terlebih dahulu di-scaling dan kemudian dikelompokkan menggunakan algoritma K-Means, menghasilkan `cluster_labels` untuk setiap penumpang.
2.  **Perhitungan Tingkat Survival per Cluster:** Untuk setiap cluster yang terbentuk, dihitung rata-rata tingkat kelangsungan hidup (`Survived`) dari penumpang yang berada dalam cluster tersebut (hanya menggunakan data pelatihan yang memiliki label `Survived`). Ini menghasilkan `cluster_actual_survival_means`.
3.  **Binerisasi:** Berdasarkan `cluster_actual_survival_means`, sebuah ambang batas (threshold, dalam kasus ini 0.5) ditetapkan. Jika rata-rata tingkat survival suatu cluster lebih besar dari 0.5, cluster tersebut diasumsikan memiliki 'potensi survival tinggi' dan diberi label `1`. Sebaliknya, jika rata-rata tingkat survival kurang dari atau sama dengan 0.5, cluster tersebut diberi label `0` sebagai 'potensi survival rendah'.
4.  **Pemetaan ke `y_cluster_logic`:** Setiap penumpang kemudian diberi nilai `y_cluster_logic` sesuai dengan label biner dari cluster tempat mereka berada.

**Tujuan `y_cluster_logic`:**

`y_cluster_logic` berfungsi sebagai alternatif atau tambahan target variabel. Dengan mengubah target `Survived` yang asli (yang bisa jadi kompleks) menjadi target biner yang lebih 'terdefinisi' oleh karakteristik cluster, kita dapat:

*   **Menyederhanakan Masalah:** Kadang-kadang, mengelompokkan data terlebih dahulu dan kemudian memprediksi karakteristik kelompok dapat menyederhanakan masalah klasifikasi.
*   **Menangkap Pola Tersembunyi:** Cluster dapat mengungkap segmen-segmen penumpang dengan karakteristik survival yang berbeda yang mungkin tidak langsung terlihat dari fitur-fitur individu.
*   **Evaluasi Model Regresi:** Karena `y_cluster_logic` adalah biner (0 atau 1), model regresi dapat dilatih untuk memprediksi probabilitas menjadi bagian dari 'cluster survival tinggi' atau 'cluster survival rendah'.

**Observasi dari `y_cluster_logic`:**

Seperti yang terlihat dari output sebelumnya, nilai rata-rata `y_cluster_logic` di `df_train` adalah sekitar 0.1156. Ini menunjukkan bahwa hanya sekitar 11.56% dari penumpang dalam data pelatihan yang dikelompokkan ke dalam kategori 'tinggi kemungkinan survival' (nilai 1) berdasarkan logika cluster yang diterapkan. Distribusi `y_cluster_logic` adalah sebagai berikut:

*   **0 (rendah kemungkinan survival):** 788 penumpang
*   **1 (tinggi kemungkinan survival):** 103 penumpang

Hal ini mengindikasikan bahwa sebagian besar penumpang di data pelatihan berada dalam cluster yang secara statistik memiliki tingkat kelangsungan hidup di bawah ambang batas yang ditetapkan.

In [ ]:
mean_y_cluster_logic = df_train['y_cluster_logic'].mean()
threshold_for_mean = 0.5 # Using the same threshold as before for comparison

print(f"Mean of y_cluster_logic in df_train: {mean_y_cluster_logic:.4f}")

if mean_y_cluster_logic > threshold_for_mean:
    print(f"The mean of y_cluster_logic ({mean_y_cluster_logic:.4f}) is above the threshold ({threshold_for_mean}).")
else:
    print(f"The mean of y_cluster_logic ({mean_y_cluster_logic:.4f}) is not above the threshold ({threshold_for_mean}).")

print("\nDistribution of y_cluster_logic in df_train:")
display(df_train['y_cluster_logic'].value_counts())

Insight dari Model Klasifikasi (Dievaluasi pada Data Pelatihan)
Dari evaluasi model klasifikasi pada data pelatihan, kita bisa melihat perbedaan kinerja yang signifikan antar model:

*   **Decision Tree:** Model ini menunjukkan akurasi pelatihan tertinggi (sekitar 0.9529). Ini berarti Decision Tree sangat baik dalam mempelajari pola pada data pelatihan. Namun, akurasi yang sangat tinggi pada data pelatihan seringkali menjadi indikasi overfitting, di mana model terlalu spesifik pada data yang dilihatnya dan mungkin tidak akan berkinerja sebaik itu pada data baru yang belum pernah dilihat. Precision, Recall, dan F1-score untuk kedua kelas juga sangat tinggi, mengonfirmasi kemampuan model dalam membedakan kelas pada data latih.
*   **SVM:** Model ini menempati posisi kedua dengan akurasi pelatihan 0.8799. SVM menunjukkan kinerja yang solid dengan keseimbangan yang baik antara precision dan recall untuk kedua kelas, menjadikannya pilihan yang cukup handal.
*   **KNN dan Logistic Regression:** Kedua model ini memiliki akurasi yang mirip, yaitu 0.8653 untuk KNN dan 0.8631 untuk Logistic Regression. Keduanya cenderung lebih baik dalam memprediksi kelas 'Not Survived' (kelas 0) dibandingkan 'Survived' (kelas 1), yang terlihat dari nilai recall kelas 0 yang lebih tinggi. Ini mungkin mengindikasikan bahwa data kelas 0 lebih mudah diidentifikasi.
*   **Gaussian Naive Bayes:** Model ini menunjukkan akurasi pelatihan terendah (0.8272). Meskipun demikian, model ini masih memberikan kinerja yang layak dan bisa menjadi baseline. Precision untuk kelas 1 (Survived) adalah yang terendah di antara model lain, menunjukkan bahwa model ini cenderung kurang akurat dalam mengidentifikasi penumpang yang selamat.

**Kesimpulan Model Klasifikasi:** Sementara Decision Tree unggul dalam 'menghafal' data pelatihan, kinerja sebenarnya pada data tak terlihat harus diperiksa lebih lanjut untuk menilai generalisasi. Model seperti SVM, KNN, dan Logistic Regression menawarkan kinerja yang lebih stabil dan cenderung lebih baik dalam generalisasi, meskipun dengan akurasi pelatihan yang sedikit lebih rendah.

Insight dari Model Regresi (Dievaluasi pada Data Validasi)
Model regresi digunakan untuk memprediksi `y_cluster_logic`, yang merupakan representasi probabilitas survival berdasarkan cluster. Evaluasi pada data validasi menghasilkan:

*   **Linear Regression:** Menunjukkan kinerja yang hampir sempurna dengan MAE, MSE, dan RMSE mendekati nol, serta nilai R-squared (R2) sebesar 1.0000. Ini mengindikasikan bahwa model linier dapat menjelaskan variansi dalam y_cluster_logic dengan sangat baik pada data validasi.
*   **Decision Tree Regression:** Juga menunjukkan kinerja yang sempurna dengan MAE, MSE, RMSE, dan R2 yang identik dengan Linear Regression. Ini tidak mengherankan karena Decision Tree dapat membagi data dengan sangat tepat hingga menghasilkan error nol pada set data yang sudah diproses secara deterministik (seperti `y_cluster_logic`).
*   **Random Forest Regression:** Model ini juga menunjukkan kinerja yang sangat kuat dengan MAE dan MSE yang sangat rendah, serta R2 yang mendekati 1.0000 (0.999951). Sedikit deviasi dari nol pada MAE, MSE, dan RMSE dibandingkan dengan Linear Regression dan Decision Tree Regression mungkin disebabkan oleh sifat ensemble dan acak dari Random Forest, tetapi secara keseluruhan, kinerja tetap luar biasa dalam memprediksi target biner yang didapat dari pengelompokan.

**Kesimpulan Model Regresi:** Hasil untuk model regresi sangat mengesankan, menunjukkan bahwa `y_cluster_logic` dapat diprediksi dengan akurasi yang hampir sempurna menggunakan fitur-fitur yang ada. Ini menggarisbawahi bagaimana representasi target yang berasal dari pengelompokan dapat diserap dengan baik oleh model-model ini. Namun, penting untuk dicatat bahwa kesempurnaan ini mungkin sebagian karena `y_cluster_logic` itu sendiri adalah target yang 'direkayasa' dari fitur yang sama, yang dapat mempermudah model untuk mempelajarinya.

## **Ringkasan Metodologi dan Hasil**

Berikut adalah ringkasan hasil untuk setiap poin metodologi yang disebutkan, berdasarkan notebook dan percakapan sebelumnya:

**1. Data Preprocessing**

*   **Menangani missing values:**
    *   `Age`: Diisi menggunakan median usia berdasarkan `Pclass` dan `Sex` untuk akurasi yang lebih baik.
    *   `Embarked`: Diisi dengan 'S' setelah investigasi mendalam (penumpang terkait diketahui embarkasi dari Southampton).
    *   `Fare`: Diisi dengan median `Fare` dari penumpang kelas 3 yang bepergian sendiri.
*   **Encoding variabel kategorikal:** Fitur-fitur seperti `Embarked`, `Sex`, `Deck`, `Title`, `Family_Size_Grouped`, `Age` (setelah binning), dan `Fare` (setelah binning) diubah menjadi numerik menggunakan `LabelEncoder`.
*   **Data cleaning:** Kolom `Cabin` dihapus karena 80% nilainya hilang, dan diganti dengan fitur `Deck` yang baru.

**2. Feature Engineering**

*   **Transformasi fitur `Cabin` menjadi `Deck`:** Fitur `Deck` dibuat dari huruf pertama kolom `Cabin`. Nilai 'T' diganti menjadi 'A', dan kemudian Dek 'A', 'B', 'C' digabungkan menjadi 'ABC'; 'D', 'E' menjadi 'DE'; dan 'F', 'G' menjadi 'FG', sedangkan 'M' (missing) tetap sebagai kategori terpisah.
*   **Pembuatan fitur baru:**
    *   `Family_Size`: Dibuat dengan menjumlahkan `SibSp`, `Parch`, dan 1 (untuk penumpang itu sendiri).
    *   `Ticket_Frequency`: Dihitung berdasarkan jumlah kemunculan setiap nilai `Ticket`.
    *   `Title`: Diekstrak dari `Name` dan dikelompokkan menjadi 'Miss/Mrs/Ms', 'Dr/Military/Noble/Clergy', 'Master', dan 'Mr'.
    *   `Is_Married`: Fitur biner (0/1) yang dibuat berdasarkan `Title` 'Mrs'.
    *   `Family_Survival_Rate` dan `Ticket_Survival_Rate`: Dibuat melalui target encoding berdasarkan nama keluarga dan nomor tiket, masing-masing, untuk keluarga/tiket yang muncul di set pelatihan dan pengujian.
    *   `Survival_Rate`: Rata-rata dari `Family_Survival_Rate` dan `Ticket_Survival_Rate`.
    *   `Survival_Rate_NA`: Menunjukkan apakah rate survival keluarga/tiket tersedia atau tidak.
*   **Pengelompokan kategori:** Fitur `Fare` dan `Age` di-binning menggunakan `pd.qcut` menjadi 13 dan 10 kuantil, masing-masing.
*   **Feature selection:** Fitur-fitur yang tidak relevan atau telah digantikan oleh fitur yang direkayasa (misalnya `Cabin`, `Name`, `Ticket`, `Parch`, `SibSp`, `Family_Size` asli, `Pclass` asli, `Sex` asli, `Embarked` asli, `Title` asli) dihapus dari set fitur akhir untuk pemodelan.

**3. Clustering**

*   **Metode:** K-Means Clustering diterapkan.
*   **Optimal K:** Metode Elbow digunakan untuk menentukan jumlah cluster optimal, yang hasilnya menunjukkan `optimal_k = 3`.
*   **Fitur untuk clustering:** Fitur yang digunakan untuk clustering (`X_cluster`) adalah `Age`, `Fare`, `Ticket_Frequency`, dan `Survival_Rate`, setelah dilakukan penskalaan dengan `StandardScaler`.
*   **Hasil:** Data dikelompokkan ke dalam 3 cluster (`cluster_labels`).

**4. Pembentukan Variabel Target (Y)**

*   `y_cluster_logic`: Variabel target biner baru ini dibuat berdasarkan hasil clustering. Untuk setiap cluster, rata-rata nilai `Survived` dari data pelatihan dihitung (`cluster_actual_survival_means`). Kemudian, jika rata-rata survival rate dari suatu cluster lebih besar dari 0.5, cluster tersebut diberi label 1 (high_survival_cluster), jika tidak, diberi label 0 (low_survival_cluster). `y_cluster_logic` kemudian merupakan pemetaan `cluster_labels` ke label biner ini.   Variabel target y_cluster_logic memiliki nilai rata-rata sekitar 0.1156. Ini berarti sekitar 11.56% dari data pelatihan dikelompokkan ke dalam kategori 'tinggi kemungkinan survival' (nilai 1) berdasarkan logika cluster. Berikut adalah distribusinya:

Nilai 0 (rendah kemungkinan survival): 788 penumpang
Nilai 1 (tinggi kemungkinan survival): 103 penumpang[teks link](https://)                                              

**5. Klasifikasi**

*   **Model yang digunakan:**
    *   Random Forest Classifier (dua versi: `single_best_model` dan `leaderboard_model`).
    *   Decision Tree Classifier.
    *   K-Nearest Neighbors (KNN) Classifier.
    *   Gaussian Naive Bayes Classifier.
    *   Logistic Regression.
    *   Support Vector Machine (SVM).
*   **Pendekatan:**
    *   Single model: Setiap model dilatih pada seluruh `X_train` dan `y_train`.
    *   Model dengan k-fold cross validation: `leaderboard_model` dievaluasi menggunakan `StratifiedKFold` dengan `N=5` split untuk menghitung OOB score, ROC AUC, dan feature importances.
*   **Evaluasi (pada data pelatihan):**
    *   Accuracy:
        *   Decision Tree: 0.9529
        *   SVM: 0.8799
        *   KNN: 0.8653
        *   Logistic Regression: 0.8631
        *   Naive Bayes: 0.8272
    *   `Confusion Matrix`: Dihasilkan dan divisualisasikan untuk setiap model.
    *   `Classification Report`: Disediakan untuk setiap model, merinci presisi, recall, dan f1-score per kelas.

**6. Regresi**

*   **Data Split:** `X_train` dan `y_train` dibagi menjadi `X_train_sub`, `X_val_sub`, `y_train_sub`, dan `y_val_sub` untuk evaluasi regresi.
*   **Model:**
    *   Linear Regression.
    *   Decision Tree Regression.
    *   Random Forest Regression.
*   **Evaluasi (pada data validasi):**
    *   Linear Regression: MAE: 0.0000, MSE: 0.0000, RMSE: 0.0000, R2: 1.0000
    *   Decision Tree Regression: MAE: 0.0000, MSE: 0.0000, RMSE: 0.0000, R2: 1.0000
    *   Random Forest Regression: MAE: 0.0002, MSE: 0.0000, RMSE: 0.0022, R2: 0.999951

**7. Perbandingan Model**

*   **Klasifikasi:** Decision Tree menunjukkan akurasi pelatihan tertinggi, tetapi SVM, KNN, dan Logistic Regression menawarkan kinerja yang lebih stabil (kemungkinan lebih baik dalam generalisasi, meskipun dengan akurasi pelatihan sedikit lebih rendah).
*   **Regresi:** Semua model regresi (Linear Regression, Decision Tree Regression, Random Forest Regression) menunjukkan kinerja yang hampir sempurna (R2 mendekati 1.0) dalam memprediksi `y_cluster_logic` pada data validasi, menunjukkan bahwa target yang direkayasa ini sangat dapat diprediksi dari fitur yang ada. Random Forest sedikit di bawah kesempurnaan mutlak namun tetap sangat akurat.

**GRAFIK CLUSTERING**

In [ ]:
import matplotlib.pyplot as plt

plt.figure()
# The error KeyError: 'cluster_labels' indicates that the DataFrame 'df' does not have this column.
# 'df_all_for_clustering' is the DataFrame that contains the 'cluster_labels'.
# Also, 'Age' and 'Fare' have been binned and label-encoded in previous steps.
# To ensure the plot runs without error, we'll use df_all_for_clustering.
plt.scatter(df_all_for_clustering['Age'], df_all_for_clustering['Fare'], c=df_all_for_clustering['cluster_labels'])
plt.xlabel('Age (Binned and Encoded)')
plt.ylabel('Fare (Binned and Encoded)')
plt.title('Clustering Result (Age vs Fare - Binned and Encoded)')
plt.savefig('cluster_plot.png')
plt.show()

**GRAFIK PERBANDINGAN MODEL**

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

hasil = pd.DataFrame({
    'Model': ['Decision Tree','SVM','KNN','LogReg','Naive Bayes'],
    'Accuracy': [0.9529, 0.8799, 0.8653, 0.8631, 0.8272]
})

plt.figure()
plt.bar(hasil['Model'], hasil['Accuracy'])
plt.xticks(rotation=45)
plt.title('Perbandingan Model Klasifikasi')
plt.savefig('model_comparison.png')
plt.show()

**GRAFIK REGRESI**

In [ ]:
import matplotlib.pyplot as plt

model = ['Linear','Decision Tree','Random Forest']
r2 = [1.0, 1.0, 0.999951]

plt.figure()
plt.bar(model, r2)
plt.title('Perbandingan Model Regresi (R2)')
plt.savefig('regression_plot.png')
plt.show()

In [ ]:
print('cluster_plot.png')
print('model_comparison.png')
print('regression_plot.png')

In [ ]:
from google.colab import files
files.download('cluster_plot.png')

In [ ]:
from google.colab import files
files.download('model_comparison.png')

In [ ]:
from google.colab import files
files.download('regression_plot.png')

In [ ]:
acc_dt = df_training_accuracies[df_training_accuracies['Model'] == 'Decision Tree']['Accuracy'].iloc[0]
acc_svm = df_training_accuracies[df_training_accuracies['Model'] == 'SVM']['Accuracy'].iloc[0]
acc_knn = df_training_accuracies[df_training_accuracies['Model'] == 'KNN']['Accuracy'].iloc[0]
acc_lr = df_training_accuracies[df_training_accuracies['Model'] == 'Logistic Regression']['Accuracy'].iloc[0]
acc_nb = df_training_accuracies[df_training_accuracies['Model'] == 'Naive Bayes']['Accuracy'].iloc[0]

hasil = pd.DataFrame({
    'Model': ['Decision Tree','SVM','KNN','LogReg','Naive Bayes'],
    'Accuracy': [acc_dt, acc_svm, acc_knn, acc_lr, acc_nb]
})

In [ ]:
!pip install nbformat
!jupyter nbconvert --ClearMetadataPreprocessor.enabled=True --inplace *.ipynb